<h1 style="font-size:250%">Gummybear Tomography: Particle Localization in Translucent Simulated Phantoms</h1>

This document is an executable Jupyter notebook that reproduces all the calculations necessary for this project.

The primary report, as an .md file, is available at [GummyBearTomography_Summary.md](GummyBearTomography_Summary.md)

# 0. Before you start: installation, setup

This project combines deep learning experiments (PyTorch) with optical diffusion simulations (Netgen/NGSolve). While all code included in the repository was used to generate the results reported in this notebook, reproducing the complete environment may require additional setup depending on the operating system and hardware configuration.

The project was developed and tested using Python 3.12 on an Apple Silicon M2 system, using Metal GPU support for PyTorch. During development, newer Python versions (3.13 and 3.14) resulted in compatibility issues with parts of the software stack and were therefore not used for the reported experiments. Compatibility on other operating systems and Python versions was not systematically evaluated.

A typical environment configuration sequence from repo root is:

```bash
python3.12 -m venv ./venv
source ./venv/bin/activate # Windows: .\venv\Scripts\activate 
pip install -r requirements.txt    
```




### Package installation from repo

In [ ]:
from __future__ import annotations
from pathlib import Path


# Install the library from the local repo

# First get path. If for any reason, we are not at repo root, move up until "pyproject.toml" is found
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
# Knowing the repo root, we can install the libraries in the venv    
!pip install --quiet --no-cache-dir "{ROOT}[fem]" -c "{ROOT}/requirements.txt"

# If FEM package installation is difficult, you can try dl (deep learning) + dev tools (dev):
# !pip install --quiet --no-cache-dir "{ROOT}[dl,dev]" -c "{ROOT}/requirements.txt"
# In that case, keep the global DATA_MODE = "inspect" (set just after Imports).


### Reproducibility mode (`DATA_MODE`)

Optical simulations and PyTorch training in `"full"`mode take time. The `"demo"` shortens execution time considerably, while `"inspection"`skips the optical pipeline integrally and displays instead pre-generated, expected output.

| Mode | Role |
|------|------|
| `"inspect"` | Use packaged / on-disk artifacts only (no optical generation; pre-packaged figures where optical simulation is necessary). |
| `"demo"` | Small regenerable demo corpora + live optical illustration where applicable. First run: ~ 10-20 minutes for dataset preparation part, then cached|
| `"full"` | Full workbooks / corpora + live optical illustration where applicable. First run: ca. 2h with 10 core parallel processing; then cached  |

Default: `"full"`, be careful to download data from Hugginface beforehand unless you want to re-run the full generation pipeline. Otherwise choose `"demo"`, or `"inspect"` as fallback when Netgen/NGSolve do no run.  

### ML checkpoint mode (`READ_CHECKPOINTS`)

Slow localisation studies (learning-rate sweeps, full train→val/test, multi-view fusion) write checkpoints under `checkpoints/<milestone>/` (e.g. `checkpoints/m8/m08_learning_rate_study.pt`). **Checkpoint read/write is only enabled when `DATA_MODE=="full"`**, so demo/inspect runs cannot overwrite the full-scale artifact. When `READ_CHECKPOINTS=True` and a checkpoint is present, those cells **load** it instead of retraining. When `False`, full-mode studies re-run and **overwrite** the checkpoint.

The checkpoints and the integral optical data from a full run is on Huggingface, at https://huggingface.co/datasets/tbhugging/gummybear-tomography/tree/main

If you do not want to run the full simulation and ML pipeline (12h+):
- Get the checkpoints folder from the Huggingface repository, and add it at the repo root
- Get the data folder from the Hugging Face repository, and add it at repo root. Attention, within the data folder, in data/generated/ on Hugginface, there are zips which will need to be de-zipped.

 




In [ ]:
# ---------------------------------------------------------------------------
# Global optical mode switch — edit this one line for the whole notebook
# ---------------------------------------------------------------------------
DATA_MODE = "full"  # "inspect" | "demo" | "full"
# ---------------------------------------------------------------------------
# When True, ML study cells load existing checkpoints if present (no overwrite).
# When False, studies retrain and overwrite checkpoints.
READ_CHECKPOINTS = True
# ---------------------------------------------------------------------------

_ALLOWED_MODES = {"inspect", "demo", "full"}
if DATA_MODE not in _ALLOWED_MODES:
    raise ValueError(
        f"DATA_MODE must be one of {sorted(_ALLOWED_MODES)}, got {DATA_MODE!r}"
    )
print(f"DATA_MODE={DATA_MODE}")
print(f"READ_CHECKPOINTS={READ_CHECKPOINTS}")


### Imports

In [ ]:
# imports
import shutil
import tempfile
from dataclasses import replace
from importlib.resources import as_file, files
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

import math

from gummybear.datasets import (
    DEFAULT_SPLIT_FRACTIONS,
    randomize_workbook_particle_centers,
    randomize_workbook_sequence_splits,
    run_generation_workbook,
)
from gummybear.geometry import load_stl
from gummybear.optics import (
    OpticalMaterialConfig,
    PointLightConfig,
    SourceSamplingParams,
    deposit_ray_source,
    generate_diffusion_mesh,
    in_object_segments_from_rays,
    make_source_ray_bundle,
    refract_ray_bundle,
    sample_diffuse_image,
    solve_diffusion,
)
from gummybear.particles import (
    ParticleSet,
    ParticleSphere,
    build_affected_transport_pairs,
    compute_transport_source_correction,
    intersect_segments_with_particles,
)
from gummybear.paths import display_path, repo_relative_path
from gummybear.rays import PinholeCameraConfig, first_visible_hits_with_points, make_camera_rays
from gummybear_validation.plotting import (
    plot_active_element_scalar_3d,
    plot_refractive_illumination_scene,
    source_active_element_mask,
)
from tomography_ml.gummybear_data_catalog import (
    HierarchicalCameraLightDataset,
    apply_image_normalize,
    build_catalog_rows,
    build_illumination_joint_groups,
    catalog_jobs_to_dataframe,
    count_groups_by_split,
    groups_for_split,
    load_catalog_jobs,
)
from tomography_ml.gummybear_data_catalog.task_dataset import DatasetTaskSpec, build_task_dataset
from tomography_ml.localization.localize_multiview import (
    M10_LIGHT_ANGLES_DEG,
    light_angle_deg_from_optical_setup_id,
)
from tomography_ml_validation.plotting import (
    display_anomaly_camera_light_grid,
    display_anomaly_still_vs_orbit,
)


# 1. Problem statement



Many imaging tasks require estimating the three-dimensional position of objects that are embedded within a partially transparent volume while only a limited number of observations are available. This can for instance concern localization of particles or markers in turbid media such as fogs, clouds, emulsions, suspensions, tissues, and hydrogels. The same problem appears in microscopy, biomedical imaging and other domains when only a few views of a translucent volume are available.  

Automated image analysis with the aid of artificial intelligence is increasingly applied to tomographic imaging. In information-constrained settings, neural networks must infer spatial location from indirect visual cues. However, common convolutional neural network architectures often rely on global pooling operations that improve parameter efficiency at the cost of discarding spatial information. This project investigates whether physically informed spatial-frequency representations can compensate for this loss of spatial information and improve localization accuracy when observational information is scarce, while simultaneously identifying potential costs of using such priors. The aim is to derive design principles for architectures that are both precise and parameter-economical. Although not implemented in this project, application aims include mobile and possibly embedded devices.

This project uses a synthetic optical tomography framework, based on a combination of refractive ray bending, scattering energy deposition and finite-element simulation for diffusion of light in a gummybear phantom. Based on the synthetic, fully controlled dataset obtained, the study evaluates the usefulness of physically informed spatial representations in the setting of the information-scarce case of positional inference from a single camera view. The results are further compared to richer tomographic settings including multiple views and multiple light sources. The goal is to identify when physically informed representations can compensate for information scarcity and to derive design principles for accurate yet parameter-efficient localization systems.



# 2. Introduction: Background, Research Aim and Approach

The project aim is to investigate whether physically informed spatial-frequency representations can aid in the task of particle localization from synthetic images of a translucent virtual gummybear phantom, be it through increase in localization precision or increase in parameter efficiency, or both.

The central research question is therefore whether physically informed spatial-frequency representations can compensate for spatial information lost during pooling, particularly when only limited observations are available.

## Background and Hypothesis

### Tomographic reconstruction

The project falls mainly in the domain of tomographic imaging, defined as reconstruction or localization of structures from indirect observations. It is generally admitted that more viewpoints or projections provide more information and make tomographic reconstruction easier (Kak, A. C., & Slaney, M. Principles of Computerized Tomographic Imaging, 1988: https://ia801905.us.archive.org/33/items/CTMRI/Principles%20of%20Computer%20Tomography%20Imaging.pdf). Similarly, the use of multiple light sources can improve reconstruction quality (Gibson, Hebden & Arridge. Recent advances in diffuse optical imaging, 2005: https://doi.org/10.1088/0031-9155/50/4/r01).

The extreme case of a single observation is particularly challenging because depth and spatial position must be inferred from indirect image cues rather than from multiple independent projections. Such settings are therefore substantially more underdetermined than conventional tomographic acquisition with multiple views (Kak, A. C., & Slaney, M. Principles of Computerized Tomographic Imaging, 1988). Recent advances in deep learning have shown that neural networks can successfully perform localization and reconstruction tasks from incomplete or indirect observations, making them attractive tools for challenging inverse imaging problems (Ongie et al., Deep Learning Techniques for Inverse Problems in Imaging, 2020: https://arxiv.org/abs/2005.06001). It is therefore a particular objective of this project to also address the single-view case in the context of deep learning.


### Spatial encoding and project hypothesis

For tasks related to spatial information different encoding strategies have been chosen in the literature. In the classical case of CNN-based image treatment, we can consider the following spatial information flow:

Image -> CNN features (x,y) -> Spatial aggregation -> Head -> coordinate prediction.

Simplifying technical detail, the spatial aggregation step can be classified as a function of the spatial information retained. Schematically, some prime literature examples ordered according to spatial information conservation vs. model complexity (specified as parameter count):

| Approach | Spatial Information Transfer Mode | Impact on Parameter Count | Reference |
|---------|---------|---------|---------|
| Global Average Pooling (GAP) | Averaging over (x,y) discards all explicit spatial information | Minimal parameter count | Lin, M., Chen, Q., & Yan, S. "Network In Network", 2013. https://arxiv.org/abs/1312.4400 |
| DeepSets | No explicit spatial information, but learned permutation-invariant aggregation of local descriptors| Moderate parameter count | Zaheer, M. et al. "Deep Sets", NeurIPS 2017.  https://arxiv.org/abs/1703.06114 |
| Flatten | All spatial information retained in ordering | Large parameter count | Krizhevsky, A., Sutskever, I., Hinton, G. "ImageNet Classification with Deep Convolutional Neural Networks", 2012. https://papers.nips.cc/paper_files/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf |

Table 1. Spatial Information Transfer Modes 





### Fourier-inspired spatial encoding

In Table 1, one can see a broad correlation between model size (parameter count) and retention of spatial information when no special measures are taken. 

With the advent of transformers, explicit efforts were made to encode positional information without substantially increasing model size. In their seminal transformer paper, Vaswani et al. (2017, https://arxiv.org/abs/1706.03762) introduced sinusoidal positional encodings, in which sine and cosine functions of different frequencies are added to token embeddings to provide position information while introducing no additional trainable parameters. Rotary Positional Embeddings (RoPE) are a later approach that similarly exploits sinusoidal structure, but encodes position through rotations in embedding space rather than additive positional vectors (Su et al., 2021, https://arxiv.org/abs/2104.09864). In RoPE-based transformers, the dot product between query and key vectors converts these rotations into relative positional information, causing attention scores to depend on the relative displacement between tokens rather than their absolute positions. These works illustrate the broader usefulness of Fourier-inspired basis functions for compact positional and spatial representations, and provide examples of both additive and multiplicative uses of Fourier terms.

Beyond spatial encoding, Fourier-based approaches offer an interesting opportunity for frequency modulation. This has explicitly been advocated by Tancik et al. (2020, https://arxiv.org/abs/2006.10739). Doing so, they successfully modulate the spectral response towards higher frequencies otherwise difficult to fit in image treatment, with successful application for instance in image sharpening. Interestingly, however, they apply Fourier transformation not in the embedding, but the coordinate space.

It is interesting to reconsider Table 1 in the light of spatial frequency content. GAP, by averaging, has the same effect as applying the constant, 0-th order Fourier term and is thus the low limit representation of spatial frequency transfer. Flattening retains all the spatial information and thus potentially the full set of spatial frequencies. 

DeepSets occupies an intermediate position in model size and spectral expressiveness. Unlike simple averaging, which applies a fixed aggregation rule, the learned linear transformation applied to embedding vectors at the heart of the DeepSet architecture can reweight and mix components of the representation. Indeed, when viewed in the Fourier basis, a general linear transformation couples frequency components rather than treating them independently (Oppenheim et al., 1997). Consequently, DeepSets can emphasize or suppress specific spatial frequency patterns prior to aggregation, providing greater spectral expressiveness than simple averaging.

I here anticipate that intentionally including limited modes at higher spatial frequency will allow improvements in localization tasks with negligible increase in parameter counts. Using approaches inspired by Tancik (2020, https://arxiv.org/abs/2006.10739) this can be done without, or with limited model size increase. Given that fully dense heads can transfer the entire spatial frequency spectrum, the approach seems particularly useful when model size is a constraint, such as in embedded or to some extent in model or real-time deployment. In this context, a cautionary project hypothesis is:

***
**Fourier-based low-spatial-frequency representations are most useful for tomographic particle localization when spatial information diversity is limited.**
***

Exploration of the low-spatial frequency domain, as opposed to high spatial frequencies as explored by Tancik et al., is physically motivated: In diffuse optical systems, scattering tends to attenuate higher spatial frequencies, making low-frequency spatial representations particularly relevant for localization from indirect optical observations.


## Relative and Absolute Positional Encoding with Fourier wavelets

Summarizing, Fourier representations have been incorporated into various neural networks. Two fundamentally different objectives relevant to this project are:
- Vaswani et al. (https://arxiv.org/pdf/1706.03762) employ sinusoidal functions as additive, relative positional encodings in self-attention; this is a specific case of Rotary Positional Embeddings.
- Tancik et al. (https://arxiv.org/abs/2006.10739) transform coordinates into a Fourier feature representation for enhanced high-frequency processing. 

While these approaches are both key conceptual inspirations for this project, they facilitate learning of **relative positional relationships** or **high-frequency functions**. The goal of the present work is different: preserving **absolute spatial information** during feature aggregation for particle localization tasks, at **low spatial frequencies** as appropriate for light diffusion in partially opaque phantoms.

In order to illustrate concretely the role of spatial frequencies, consider a CNN feature map containing a strongly localized activation corresponding to a particle. Global Average Pooling (GAP) preserves the presence of that activation but discards its location; the same average value is obtained regardless of particle position. At the opposite extreme, flattening preserves the complete spatial representation at substantial parameter cost. The key idea explored here is that Fourier pooling provides an intermediate representation, transforming spatial position into a characteristic "ripple" pattern of channel activations permitting to easily reconstruct absolute position while retaining a compact representation.

## Model Architecture

The experiments employ a common backbone with a CNN → spatial aggregation → MLP design, allowing direct assessment of the impact of GAP, Fourier Pooling, and Flattening. The canonical architecture is defined with batches of image of height $H$ and width $W$ as the input; variants with multiple views and other architectural variants are employed to address specific scientific or optimization questions and shall be discussed in detail later on (Dataset Generation).

```text
┌─────────────────┐
│   Input Image   │
└─────┬───────────┘
      │
      ▼
┌───────────┐
│    CNN    │
└─────┬─────┘
      │ C×H×W
      ▼
┌──────────────────┐
│ Feature Maps     │
└─────┬────────────┘
      │                              C
      ├─────────────── GAP ───────────────► MLP ► Position
      │                              C
      ├──────────── Fourier Pool ─────────► MLP ► Position
      │
      └───────────── Flatten ─────────────► MLP ► Position
                                    F=(C×H×W)
```
Note that for simplicity, batching is ignored in the scheme, the actual tensor dimension after the CNN is $[B,C,H,W]$, with $B=1$, $B=16$ or $B=32$ depending on the exact example. The extreme cases of GAP and full flattening are indicated, but for comparison, a DeepSet-architecture will also be used on some experiments with a learned permutation-invariant spatial aggregation step.


The canonical tensor sizes (again ignoring batching):
 
| Branch | Vector size |
|---------|---------|
| GAP | $C = 64$ (each channel from map averaging) |
| Fourier pool | $C = 64$ (one fixed Fourier mode per channel) |
| Flatten | $F = H \cdot W \cdot C = 128 \cdot 128 \cdot 64 = 1\,048\,576$  |

Note that as part of the coordinate head, some MLP involve expansion to a maximum of 128 channels. Also note that, as already mentioned above, in the case of multi-view analysis, variant modes are also explored. This includes namely the repeated application of single-view analysis as outlined above followed by estimate averaging or embedding fusion and alternatively direct ingestion of $V$ x $H$ x $W$ tensors with multiple views $V$.

### Mathematics of the spatial aggregation layer

The mathematical relations describing the three spatial aggregation branches are:

| Readout | Operation |
|----------|----------|
| GAP | $u_c = \frac{1}{HW}\sum_{x,y} X_c(x,y)$ |
| Fourier pool | $u_c = \frac{1}{HW}\sum_{x,y} X_c(x,y)B_c(x,y)$ |
| Flatten |  $\mathbf{u} = \mathrm{vec}(X) \in \mathbb{R}^{F}$ |

where $x,y$ are the image coordinates, $X_c(x,y)$ is the activation in the $c^{\text{th}}$ channel of the feature map $X$ produced by the CNN at coordinates $(x,y)$. GAP is a simple arithmetic average over the pixel representation, while flattening corresponds to concatenation of all $F = H \cdot W \cdot C$ available feature values at all pixels into a single vector.




### Fourier pooling






Fourier pooling, the spatial pooling mode of specific interest here, is accomplished by element-wise multiplication of the feature maps $X(x,y)$ produced by the CNN with $B_c(x,y)$ functionals, before actual averaging, e.g. $u_c = \frac{1}{HW}\sum_{x,y} X_c(x,y)B_c(x,y)$ as indicated above. The $B_c$ functionals are standard Fourier cosine and sine terms, defined in detail as follows:

- $\phi_c(x,y)$ is the phase angle in channel $c$ at position $x,y$:
$$
\phi_c(x,y)
=
2\pi
\left(
k_h(c)\frac{x}{W-1}
+
k_v(c)\frac{y}{H-1}
\right)
$$
Frequency pairs $(k_h,k_v)$ are integers assigned in order of increasing total degree $k_h+k_v$, with every non-DC mode appearing twice: $(0,0),$  $(1,0),(1,0),$  $(0,1),(0,1),$  $(2,0),(2,0),$  $(1,1),(1,1)$,  $(0,2),(0,2),$  $...$
- and $B_c(x,y)$ is:

$$
B_c(x,y)
=
\begin{cases}
1,
& c=0,
\\[4pt]
\cos\!\bigl(\phi_c(x,y)\bigr),
& c\geq 1, \text{odd},
\\[4pt]
\sin\!\bigl(\phi_c(x,y)\bigr),
& c\geq 1, \text{even},.
\end{cases}
$$

For the first channel $c=0$, $B_c(x,y) \equiv 1$ such that the 0-th order Fourier channel reduces to the corresponding GAP value. Note that 0-based indexing is used in accordance with Python conventions. In terms of implementation, only the Fourier Pooling layer was implemented specifically for this project. The CNN backbone, Global Average Pooling (GAP), Flatten operation, and MLP head were implemented using standard PyTorch modules.

# 3. Gummybear phantom: A simplified, high throughput optical simulation

## Optical Simulation Pipeline

The localization experiments are trained on synthetic camera images of a translucent gummybear phantom. The specific aim of this project is to address spatial localization capacity in translucent samples, and the aim of the optical simulation pipeline is to enable rapid generation of image dataset with exactly known labels.

A simplified optical model was intentionally chosen to prioritize dataset throughput and exact label availability over physical realism. Since the aim of this project is to compare neural network architectures under controlled conditions, the simulation emphasizes reproducibility, exact label availability, and computational throughput rather than exhaustive physical realism.

The optics pipeline is as follows:

- Step 1: Geometric ray optics.
   - Generate ray bundle originating from a light source (simplified: uniform bounding-box derived random sampling with inverse square energy decay with distance)
   - Bear intersection (Trimesh library), refraction (Snell) and particle intersection (analytical)
- Step 2: Energy deposition. 
   - Netgen-derived volumetric mesh from known surface mesh.
   - Ray intersection with volumetric mesh tetrahedrons (Trimesh)
   - attenuation on these mesh intersection trajectories and the geometric (analytical) particle intersection (Lambert-Beer attenuation, scattering for deposition)
   - Deposition as source term for volumetric diffusion simulation (i.e. energy lost from ray segments per tetrahedron volume)
- Step 3: Energy diffusion
   - finite element simulation variational formulation handled by NGSolve.
   - extraction of optical energy flow from NGSolve solution at surface triangle barycenters
- Step 4: Use camera raybundle to collect image intensities (pinhole-camera). 
   - Visibility is simplified to first surface mesh hit.
   - Optical energy flow at hit points is calculated from encompassing or nearby volumetric tetrahedron (interpolation or slight extrapolation from node intensities).

### Step 1 Geometric ray optics and Step 2 energy deposition

Left: Step 1 illustrated on a gummybear phantom mesh (`cad/proto_bear_head.stl`). Rays generated from a point light (random sampling), with one analytic spherical particle. Orange segments are exterior rays (source→mesh); blue segments are Snell-refracted in-object chords. Green / cyan markers are particle entry / exit (mesh entry hits are not marked).

Right: Step 2 illustrated as **net particle-induced source delta** on the coarse diffusion mesh derived from the same surface mesh. Active (affected) tet centroids colored by $\Delta E_{\mathrm{transport}}$ (calculated as Lambert-Beer particle - background deposition plus specific local particle scatter). Interpret energy scale as relative: inverse square law and Lambert-Beer attenuate rays, lowering absolute magnitude of particle-attributable signal concomitantly.


In [ ]:
# In inspection mode, we only display a pre-made png for illustration. In demo and full mode, we generate it
# with the actual optical pipeline
_STEP12_ILLUSTRATION = (
    files("gummybear_validation")
    / "illustrations"
    / "step1_geometric_ray_optics_step2_energy_deposition.png"
)

if DATA_MODE == "inspect":
    # Packaged static figure — no Netgen/NGSolve required.
    with as_file(_STEP12_ILLUSTRATION) as _illust_path:
        display(Image(filename=str(_illust_path)))
    print(
        "DATA_MODE=inspect: showing packaged Step 1 / Step 2 illustration "
        f"({_STEP12_ILLUSTRATION.name}). "
        "Set global DATA_MODE to 'demo' or 'full' (top of notebook) to re-run the optical simulation."
    )
elif DATA_MODE in ("demo", "full"):
    # Load the bear STL (designed by yours truly in FreeCAD) 
    stl = ROOT / "cad" / "proto_bear_head.stl"
    surface_mesh = load_stl(stl)
    # Use the Netgen library to generate a coarser volumetric mesh from the fine surface mesh.
    # The availability of a volumetric mesh is an absolute requirement for the diffusion solver (Steps 3 and 4)
    diff_mesh = generate_diffusion_mesh(stl, target_elements=1000)

    # Netgen installation can be finicky, so check that the mesh is actually live. 
    assert diff_mesh.netgen_mesh is not None, "Need a live Netgen mesh for deposition."

    # Centroid to define sensible positions for particle, light, and camera.
    lo, hi = np.asarray(surface_mesh.bounds, dtype=float)
    centroid = 0.5 * (lo + hi)

    # Shared upright side view (z up) for the visualization panels.
    view_elev, view_azim = 0.0, -90.0

    # Point light source
    light = PointLightConfig(
        position=(centroid[0] + 15.0, centroid[1] + 15.0, hi[2] + 15.0),
        intensity=1.0,
    )
    # Optical material properties for the gummybear - more or less realistic for a translucent, 
    # mostly scattering object on the mm to cm scale (units: mm-1 for the absorption and scattering coefficients)
    material = OpticalMaterialConfig(
        n_refractive=1.33,
        mu_absorption=0.1,
        mu_scatter=0.3,
    )

    # Step 1: Geometric ray optics. We start with the light source.
    # Note here that they are random-sampled in a specific way: all rays originate from the point source,
    # and are defined to travel in the direction of a target point, the latter being uniformly distributed in the 
    # bounding box of the mesh. This scheme cheaply generates a distributed light source aimed at the mesh.
    #  
    # This does not yield a radiometrically uniform point source with equal radiation energy in all space directions.
    # Note however that while randomization of rays is important, spatially uniform radiation is not. Indeed spatial 
    # uniformity is required neither for ML dataset generation
    # nor in real life where as general rule, light sources are all but radiometrically uniform.


    source_rays = make_source_ray_bundle(
        light,
        surface_mesh.bounds,
        SourceSamplingParams(n_rays=512, seed=0),
    )
    # refraction at the gummy bear surface.
    entry = refract_ray_bundle(
        surface_mesh,
        source_rays,
        n_from=1.0,
        n_to=material.n_refractive,
    )
    # For rays hitting the gummy bear, define single segment from entry to exit point.
    segments = in_object_segments_from_rays(surface_mesh, entry.rays)
    # Particle positoin and optical properties. Here, at the centroid; will be varied in ML datasets.
    particle = ParticleSphere(
        center=centroid,
        radius=2.5,
        mu_abs=0.0,
        mu_scat=1.0,
        particle_id="report_demo",
    )
    # The optical simulation is designed to be able to handle multiple particles, but this capacity is not 
    # essential for the current ML project; we still have to constitute a particle set from the single particle of interest.
    particles = ParticleSet.from_particles([particle])
    # Analytical intersection between the geometrically exactly defined intra-gummybear ray segments and 
    # the also exactly defined particle(s).
    particle_events = intersect_segments_with_particles(
        segments.starts,
        segments.ends,
        particles,
    )

    # Step 2: Scene and geometric rays defined, we can now start with evaluation of optical energy deposition.

    # First: Deposit volumetric source for undisturbed rays. This is the particle-free background, or "clean" source.
    clean_source = deposit_ray_source(
        diff_mesh,
        segments,
        mu_s=material.mu_scatter,
        mu_a=material.mu_absorption,
    )
    E_clean_elem = np.asarray(clean_source.E_scat_elem, dtype=float)

    # Second: Impact of particle(s).
    # The fundamental unit for keeping track of the effect of a particle on the energy deposited by a ray is the "transport pair".
    # For each ray segment within the gummybear that hits a least one particle, a transport pair is defined.
    # The transport pair will then be used when computing the energy delta due to the particle or in the multi-particle case,
    # the cumulative effect due to all particles encountered (see compute_transport_source_correction, below). 
    # The reason for this type of path-wise book-keeping is that 
    # it lends itself more naturally not only to multi-particle scenarios but also to extensions 
    # such as ray path alteration by refractive particles
    pair_result = build_affected_transport_pairs(
        segments,
        particles,
        events=particle_events,
        mu_s=material.mu_scatter,
        mu_a=material.mu_absorption,
    )
    # The source energy deposition delta is evaluated and summed up from the transport pairs
    source_delta = compute_transport_source_correction(
        diff_mesh,
        pair_result,
        E_clean_elem,
        mu_s=material.mu_scatter,
        mu_a=material.mu_absorption,
    )

    # For drawing, source-delta and mask (for intensity coloring and masking to non-zero elements)
    delta_E_transport_elem = np.asarray(source_delta.delta_E_transport_elem, dtype=float)
    active_mask = source_active_element_mask(
        E_clean_elem,
        np.asarray(source_delta.E_particle_elem, dtype=float),
        np.asarray(source_delta.delta_E_background_elem, dtype=float),
        np.asarray(source_delta.delta_E_particle_scat_elem, dtype=float),
        delta_E_transport_elem,
    )

    # --- Side-by-side figure for steps 1 and 2 ---
    fig = plt.figure(figsize=(14, 6.5))

    ax_rays = fig.add_subplot(1, 2, 1, projection="3d")
    plot_refractive_illumination_scene(
        ax_rays,
        surface_mesh,
        light,
        source_rays,
        entry,
        segments,
        particle=particle,
        particle_events=particle_events,
        max_rays_to_draw=48,
        view_elev=view_elev,
        view_azim=view_azim,
        title="Step 1: Geometric ray optics",
    )

    ax_delta = fig.add_subplot(1, 2, 2, projection="3d")
    sc = plot_active_element_scalar_3d(
        ax_delta,
        diff_mesh,
        delta_E_transport_elem,
        "Step 2: Energy deposition",
        active_mask=active_mask,
        signed=True,
        cmap="coolwarm",
        relative_floor=1e-5,
        view_elev=view_elev,
        view_azim=view_azim,
        mesh_face_alpha=0.13,
        mesh_edge_color="0.60",
        mesh_linewidth=0.20,
        point_size=42,
    )
    if sc is not None:
        fig.colorbar(sc, ax=ax_delta, shrink=0.72, pad=0.08, label="dE total")

    plt.tight_layout()
    plt.show()

    print(
        f"source rays={source_rays.n_rays}, "
        f"refracted={entry.n_refracted}, "
        f"particle hits={len(particle_events)}, "
        f"nonzero ΔE elems={int(np.count_nonzero(delta_E_transport_elem))}"
    )
    print("Note: Only a sparse set of rays is used here for figure clarity.")

if DATA_MODE=="inspect":
    print("!!!!!  Packaged image, not evaluted here  !!!!!!!!!")


### Step 3 Energy diffusion and Step 4 camera anomaly

Left: Step 3 — Energy diffusion. The simulation solves the steady-state, isotropic diffusion of light intensity from the volumetric sources defined in Step 2, showing how optical fluence differential is spread from the particle and its shadow across the gummybear phantom. Tet centroids are colored by the fluence delta $\Delta\Phi = \Phi^{\mathrm{particle}}-\Phi^{\mathrm{background}}$.

Right: Step 4 — Image collection. The pipeline simulates a pinhole camera by inverse ray-tracing from the camera to first-line surface hits on the surface mesh. At the hit points, it samples optical flux attributable to the particle presence as $\Delta\Phi$. The intensity scale shows **per-image z-score** normalisation, corresponding to the input to ML.


In [ ]:
_STEP34_ILLUSTRATION = (
    files("gummybear_validation")
    / "illustrations"
    / "step3_energy_diffusion_step4_camera_anomaly.png"
)

if DATA_MODE == "inspect":
    # Packaged static figure — no Netgen/NGSolve required.
    with as_file(_STEP34_ILLUSTRATION) as _illust_path:
        display(Image(filename=str(_illust_path)))
    print(
        "DATA_MODE=inspect: showing packaged Step 3 / Step 4 illustration "
        f"({_STEP34_ILLUSTRATION.name}). "
        "Set global DATA_MODE to 'demo' or 'full' (top of notebook) to re-run the optical simulation."
    )
elif DATA_MODE in ("demo", "full"):
    # Step 3: Diffusion simulation.
    # There are two settings which are not immediately evident from the geometry and material optical property already defined
    # for the gummybear in step 1/2.
    # 
    # The first property is "extrapolation_length", also known as Robin leakage length.
    # This parameter is given in mesh units (e.g. mm) and is a 
    # prototypical optical diffusion simulation parameter. It represents the difficulty of light to 
    # escape from the phantom. A Robin length of the same order of magnitude as the phantom size
    # means that some light energy remains emprisoned but some details can escape for downstream analysis.
    extrapolation_length = 5.0   


    # The simulation also needs an apparent light diffusion coefficent. This is related 
    # to the gummy bear optical properties, i.e.
    # absorption and scattering coefficients, but since it is implies explicit specification of the hypothesis of fullly
    # isotropic diffusion, we calculate here via a python property which internally includes the anistropy index g, here 
    # assumed to be 0.
    D = material.diffusion_coefficient 

    # Source terms. Clean refers to phantom without particle, that is background source without partice contribution
    # The source terms are extracted from the Step 2 objects which computed the effect of the particle on optical energy deposition
    # and form the input for the diffusion PDE solver.
    S_clean = np.asarray(clean_source.S_clean, dtype=float)
    S_particle = np.asarray(source_delta.S_particle, dtype=float)

    # invoke ngsolve for variational solution to the isotropic diffusion equation, 
    # with Robin boundary conditions, for both clean and particle source terms.
    phi_clean = solve_diffusion(
        diffusion_mesh=diff_mesh,
        S_clean=S_clean,
        D=D,
        mu_a=material.mu_a,
        extrapolation_length=extrapolation_length,
    )
    phi_particle = solve_diffusion(
        diffusion_mesh=diff_mesh,
        S_clean=S_particle,
        D=D,
        mu_a=material.mu_a,
        extrapolation_length=extrapolation_length,
    )

    # Per-tet fluence delta (mean of corner nodes) — plotted at tet centroids.
    Phi_delta_tets = np.asarray(phi_particle.Phi_tets, dtype=float) - np.asarray(
        phi_clean.Phi_tets, dtype=float
    )

    # Step 4: pinhole camera aligned with the upright 3D view (elev=0, azim=-90 → -y)
    # Place the camera so that we can see the entire Gummybear upright
    cam_distance = 55.0
    camera = PinholeCameraConfig(
        camera_position=(
            float(centroid[0]),
            float(centroid[1] - cam_distance),
            float(centroid[2]),
        ),
        look_at=tuple(map(float, centroid)),
        up=(0.0, 0.0, 1.0),
        fov_deg=45.0,
        resolution=128,
    )

    # The camera rays back-track towards the gummy bear to the first mesh intersection point (if intersecting); 
    # the camera rays are regularly distributed 
    # to generate a HxW rectangular image, not randomized
    camera_rays = make_camera_rays(camera)
    H, W = camera_rays.sample_shape

    # not all cam rays hit the gummy bear. Here, we make use of the mask to define which rays hit the bear,
    # and for those that hit, we get actual com_points hit intersection values (NaN for the non-hitting)
    cam_valid, _cam_depth, _cam_faces, cam_points = first_visible_hits_with_points(
        surface_mesh,
        camera_rays,
    )

    # For the present ML study, we are interested in the diffusive part, not in direct ballistic ray transport
    # Clean "background" image
    clean_diffuse = sample_diffuse_image(
        diff_mesh,
        phi_clean.Phi_nodes,
        cam_points,
        cam_valid,
        (H, W),
    )
    # Image including particle
    particle_diffuse = sample_diffuse_image(
        diff_mesh,
        phi_particle.Phi_nodes,
        cam_points,
        cam_valid,
        (H, W),
    )
    # The delta is the particle-attributable part and is therefore used for the ML studies.
    # This isolates the localization problem from unrelated image reconstruction effects,
    # allowing the experiments to focus on whether particle position can be inferred from
    # the optical signal itself.
    Delta_I = np.asarray(particle_diffuse.I_diffuse, dtype=np.float32) - np.asarray(
        clean_diffuse.I_diffuse, dtype=np.float32
    )

    # The aim here is to illustrate images as will be recieved by the neural networks in the training pipeline
    # Use the normalization part of the ML dataset machinery in an anticipated manner  
    anomaly_batch = Delta_I[None, None, :, :]  # [V=1, C=1, H, W]
    anomaly_norm = apply_image_normalize(anomaly_batch, "per_image_zscore")[0, 0]
    vmax = float(np.max(np.abs(anomaly_norm))) if anomaly_norm.size else 1.0
    if vmax <= 0.0:
        vmax = 1.0

    fig = plt.figure(figsize=(14, 6.5))

    ax_phi = fig.add_subplot(1, 2, 1, projection="3d")
    sc_phi = plot_active_element_scalar_3d(
        ax_phi,
        diff_mesh,
        Phi_delta_tets,
        r"Step 3: Diffused fluence delta $\Delta\Phi$",
        active_mask=active_mask,
        signed=True,
        cmap="coolwarm",
        relative_floor=1e-5,
        view_elev=view_elev,
        view_azim=view_azim,
        mesh_face_alpha=0.13,
        mesh_edge_color="0.60",
        mesh_linewidth=0.20,
        point_size=42,
    )
    if sc_phi is not None:
        fig.colorbar(sc_phi, ax=ax_phi, shrink=0.72, pad=0.08, label=r"$\Delta\Phi$ (tet)")

    ax_cam = fig.add_subplot(1, 2, 2)
    im = ax_cam.imshow(
        anomaly_norm,
        cmap="gray",
        vmin=-vmax,
        vmax=vmax,
        origin="lower",  # camera rows increase with +up; keep ears upright
        interpolation="nearest",
    )
    ax_cam.set_title("Step 4: Camera anomaly (per-image z-score)")
    ax_cam.set_xticks([])
    ax_cam.set_yticks([])
    fig.colorbar(im, ax=ax_cam, shrink=0.85, pad=0.04, label="z-scored $I_{particle}-I_{clean}$")

    plt.tight_layout()
    plt.show()

    print(
        f"D={D:.4g}, extrapolation_length={extrapolation_length}, "
        f"camera={camera.resolution}² fov={camera.fov_deg}°, "
        f"hit pixels={int(np.count_nonzero(cam_valid))}/{H*W}, "
        f"anomaly z-score |max|={vmax:.3g}"
    )


if DATA_MODE=="inspect":
    print("!!!!!  Packaged image, not evaluted here  !!!!!!!!!")


### Limitations of the optical simulation pipeline

The optics pipeline is a compromise between realism and simplification for speed and development feasibility. Major limitations are:
- Single refraction: At most 1 refraction event is taken into account per light source ray
- Diffuse imaging only: Although the gummybear optics python package handles both diffusive and direct ray-based energy transport to the camera, only diffuse parts are considered here for particle localization. The diffuse part is the major contribution in translucent, highly scattering media of interests here.
- Single particle only: The simulation itself handles multiple, non-overlapping particles. For the scientific question of the utility of a Fourier aggregation layer for particle localization, the simpler single-particle scenario offers a clearer hypothesis testing path. 
- Optical simplifications. The major technical simplification at the level of the physics are:
  - Ballistic transport vs. Isotropic diffusion only (e.g. scalar representation of diffuse intensity, not angle-resolved). Partial anisotropic transport, secondary reflected or scattered rays are not considered. 
  - Stationary solution of the diffusion equation (no time-of-flight analysis)
  - Refraction only: This simulation is based on non-coherent optics, no constructive / destructive interference, Fresnel, Newton rings etc.
  - Pinhole camera without explicit lens effects.

Summarizing, a series of deliberate optical and design limitations were made. These simplify optical simulation considerably, enabling the generation of a clearly structured data body with:
- key label: particle position; 
- meta-data or feature depending on the experiment: camera position, lighting position. 
Also, the simplification and aggressive caching permitted to simulate a large body (hundreds to thousands of particle configurations, tens of thousands of individual views) on my personal machine in a reasonable time frame. 

The simplifications were explicitly made with a translucent, non-specular, non-coherent image mode that nevertheless produces interpretable optical signal for the particle localization elements. The one less intuitive element that was necessary in the simulation was the Robin boundary length to capture the fact the diffusive radiation captured transiently in the gummybear contributes significantly to the realistic aspect of a translucent object partially illuminated by the scattered energy (in the manner of a translucent lightbulb).

Consequently, conclusions drawn from this study should be interpreted in the context of the simulated regime considered here: translucent media, single-particle localization, non-coherent optics, and diffusion-dominated image formation. The primary focus of the study is the utility of spatial-frequency representations for localization under information-limited conditions rather than the exact realism of the optical simulator. While transfer learning to real-world images is planned as a future development direction, doing so introduces a second set of scientific questions related to simulation fidelity, domain transfer, and model generalization. These questions are distinct from the primary hypothesis tested in this project and are therefore left for future work.




# 4. Dataset Generation and Quality Assurance



## Project Implementation

Given the complexity of the combined task of optical simulation, dataset generation, and dataset consumption for testing the project hypothesis in machine learning, the project was planned and implemented as a series of practical progression steps.

| Project Step | Milestone | Dataset | Python Package |
|---|---|---|---|
| Optical simulation framework | M1-M5 | No ML dataset produced. These stages establish the physical and geometric simulation capability used later for dataset generation. | `gummybear`, `gummybear_validation` |
| Dataset-generation capability | M6-M7 | No final ML dataset produced. These stages develop the configurable workbook-driven generation pipeline, caching logic, and reusable data-generation infrastructure. | `gummybear`, `gummybear_validation`, `tomography_ml`, `tomography_ml_validation`|
| Single-view localization | M8 | **M8 fixed illumination dataset**. Single-particle localization under fixed illumination and multiple optical regimes. The corpus retains a camera orbit; M8 ML consumes a single fixed camera angle (180°). | `gummybear`, `tomography_ml`, (helpers from `tomography_ml_validation`) |
| Multi-view camera fusion | M9 | Reuses the M8 single-illumination dataset. The dataset is unchanged; the experiment uses multiple camera views to be consumed by the model. | `tomography_ml` (with M8 dataset) |
| Multi-illumination fusion | M10 | **M10 multi-illumination dataset**. Particle placements are simulated under multiple illumination directions for illumination-fusion experiments. | `gummybear`, `tomography_ml`, (helpers from `tomography_ml_validation`) |

The project was developed incrementally. M1-M5 establish the optical simulation framework, while M6-M7 extend this framework into a configurable dataset-generation capability. The resulting infrastructure is then used for the machine-learning experiments in M8-M10. Importantly, not every milestone produces a separate dataset: M9 reuses the M8 dataset and changes only the model input strategy.

## Implementation Scope Boundaries




### Software Project Boundary

This project treats foundational scientific-computing libraries such as PyTorch, Netgen, NGSolve, NumPy, and their dependencies as trusted infrastructure components.

Project-specific work therefore focuses on the selection, configuration,
parameterization, combination, and evaluation of these components rather
than reimplementation of their internal algorithms. Examples include the
choice of neural-network architectures and optimizers in PyTorch, mesh
generation strategies in Netgen, and finite-element spaces and solver
configuration in NGSolve.

My contribution is the creative use of existing building blocks, not the re-implementation or verification of their inner workings.

### AI Tool Usage Boundary

My stance on AI tool usage is simple: the questions, hypotheses, architecture ideas, modeling decisions, experimental design, debugging direction, and evaluation logic are mine. Also I implement core concepts manually to develop intuition and deep understanding. However, once that foundation exists, AI helps explore the implementation space faster. I decide where the project goes.


This project followed this stance:

The optical simulation concepts (M1-M5), dataset design (M6-M7, with later expansion in M8 and M10), dataset lazy-loading from catalogs, machine-learning fundamentals in the form of initial CNN/Fourier-pooling experiments, model heads, Euclidean loss function, training loops, evaluation strategy, elementary analysis and plotting were developed manually using public documentation, scientific literature, and notebook-first experimentation. The intent was to build precise understanding of the core simulation and machine learning mechanisms.

AI-assisted tooling was used as the project grew in size and complexity. OpenCode was used for planning support, while OpenCode and Cursor were used increasingly for implementation assistance. This support was mainly applied to software-engineering tasks such as refactoring, boilerplate generation, reusable package structure, caching, parallel execution, flexibilization of dataset derivation, and reduction of code duplication across related model and pipeline variants.

All AI-assisted code was reviewed, tested, and integrated manually. Scientific assumptions, simulation parameters, dataset definitions, model choices, and experimental conclusions are mine.

## Dataset Generation and Validity

### Dataset Generation: Standard Approach

For this project, dataset generation and shaping is performed in 4 steps:
- Step 1: **Configuration**. Creation of an Excel-based configuration file (scene, illumination, particle, camera orbit settings). A sequence corresponds to a set of views visible from an orbit of predefined camera positions, looking at a single gummybear scene with a single defined illumination source, single defined particle. The configuration includes particle position randomization and a primary training/validation/test split.
- Step 2: **Optical simulation**. Carried out sequence per sequence (optionally, parallel processing). Write images and manifests to disk, organized in per-sequence folders.
- Step 3: **Catalog**. Before use, reload Excel file as catalog for on-disk data, 1 row per sequence
- Step 4: **Dataset**. Instantiate dataset for PyTorch indexing according to ```x,y = dataset[i]``` Samples (indexed by $i$), features($x$) and labels($y$) are defined per task, see below.


The datasets are then ready for use with DataLoaders for ML pipelines.

Note that configuring Optical Simulation in Excel is a specific architectural choice, aiming at separating data generation from the code base. Loading a catalog before instantiating a dataset provides flexibility regarding the choice of features and labels for reusability in tasks beyond this project. For example, in this project, particle position is generally treated as a label (prediction target), while image data constitutes a feature used for prediction. The same dataset could equally support the corresponding forward problem, in which particle position serves as the feature(input) and image data as the label (prediction target).

As outlined already above, two datasets are produced:
- **M8 fixed illumination** Dataset: Fixed illumination, variable camera, random particle positions. Also contains 3 different gummybear optical property sets.
- **M10 variable illumination** Dataset: Variable illumination, variable camera, random particle position, fixed gummybear properties.

The different datasets are obtained using the same four-step pipeline. The primary differences are the experiment configuration defined in Step 1 and the task-specific dataset and tensor representations used in Step 4.




### M8 - Fixed Illumination Dataset

#### Step 1: Configuration including particle position and train/val/test randomization




In [ ]:
# This cell is responsible for path definitions used for M8 =============================================

# Storage locations depend on mode
DEMO_OUTPUT_ROOT = ROOT / "data" / "generated" / "m8_demo"
# Historical data output location for M8
FULL_OUTPUT_ROOT = ROOT / "data" / "generated" / "m8_1" / "single_particle" 
# Historical Excel configuration file name for M8
FULL_WORKBOOK_PATH = ROOT / "configs" / "m8" / "localization_single_particle.xlsx"
 
# For demo, use the packaged workbook resource rather than a repository copy.
DEMO_PACKAGE_RESOURCE = (
    files("tomography_ml_validation")
    / "test_data"
    / "configs"
    / "m8"
    / "m8_demo.xlsx"
) 


# Keep TemporaryDirectory objects alive for later cells (inspect catalog workbook).
_SESSION_TEMPDIRS: list[tempfile.TemporaryDirectory] = []

# Temporary (inspect) or persistent copy of the configuration workbook for demonstration of randomization (position/split)
def _copy_demo_workbook(dest: Path) -> Path:
    """Copy the installed demo workbook to ``dest`` with demo ``output_root``."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    with as_file(DEMO_PACKAGE_RESOURCE) as src:
        shutil.copy2(src, dest)
    frames = pd.read_excel(dest, sheet_name=None, engine="openpyxl")
    frames["sequences"]["output_root"] = "data/generated/m8_demo"
    with pd.ExcelWriter(dest, engine="openpyxl") as writer:
        for sheet_name, frame in frames.items():
            frame.to_excel(writer, sheet_name=sheet_name, index=False)
    return dest.resolve()


# In inspect mode, randomization is shown in a temporary copy of the Excel file, 
# the original remains authoritative with regard to the artifacts produced.
if DATA_MODE == "inspect":
    # Pristine packaged centres → temp only.
    OUTPUT_ROOT = DEMO_OUTPUT_ROOT
    RUN_GENERATION = False
    _inspect_tmpdir = tempfile.TemporaryDirectory(prefix="gummybear_m8_inspect_")
    _SESSION_TEMPDIRS.append(_inspect_tmpdir)
    WORKBOOK_PATH = _copy_demo_workbook(Path(_inspect_tmpdir.name) / "m8_demo.xlsx")
elif DATA_MODE == "demo":
    # Persistent materialization under configs/m8 for randomization and then generation
    OUTPUT_ROOT = DEMO_OUTPUT_ROOT
    RUN_GENERATION = True
    WORKBOOK_PATH = _copy_demo_workbook(ROOT / "configs" / "m8" / "m8_demo.xlsx")
else:  # Direct operation on the final generation file
    OUTPUT_ROOT = FULL_OUTPUT_ROOT
    RUN_GENERATION = True
    WORKBOOK_PATH = FULL_WORKBOOK_PATH

# Cache shared optical-simulation artifacts.
#
# Background energy deposition is reused for sequences sharing the
# same mesh and illumination configuration.
#
# Particle-delta energy deposition is reused for sequences sharing
# the same mesh, illumination configuration, and particle position.
CACHE_ROOT = OUTPUT_ROOT / "_cache"

def _output_root_has_simulations(output_root: Path) -> bool:
    """True when at least one sequence under ``output_root`` has a manifest."""
    root = Path(output_root)
    if not root.is_dir():
        return False
    for child in root.iterdir():
        if child.name.startswith("_") or not child.is_dir():
            continue
        if (child / "manifest.json").is_file():
            return True
    return False


SIMULATIONS_EXIST = _output_root_has_simulations(OUTPUT_ROOT)

print(f"DATA_MODE={DATA_MODE}")
print(f"RUN_GENERATION={RUN_GENERATION}")
print(f"WORKBOOK_PATH={display_path(WORKBOOK_PATH)}")
print(f"OUTPUT_ROOT={display_path(OUTPUT_ROOT)}")
print(f"CACHE_ROOT={display_path(CACHE_ROOT)}")
print(f"SIMULATIONS_EXIST={SIMULATIONS_EXIST}")


#### Randomization of particle position (M8)

Particle positions are sampled randomly and uniformly within the gummy bear volume. Candidate positions are drawn randomly and accepted only if they lie inside the gummy bear mesh. Positions outside the mesh are rejected, and a new candidate position is drawn until a valid position is obtained.

This corresponds to the assumption that particles are uniformly distributed throughout the gummy bear volume.

The randomized particle position assignment is generated programmatically (code below) and recorded for traceability and consistent use in the Excel workbook. Reusing the same random seed produces identical particle configurations, ensuring reproducible dataset generation.

The reason for using one global randomization of particle position assignment is that the main question here is the impact of architecture parameters under otherwise identical conditions.

In [ ]:
_PARTICLE_RANDOM_SEED = 42  # The Hitchhiker's Guide to the Galaxy
_cols = ["particle_setup_id", "center_x", "center_y", "center_z", "seed"]

# Helper to randomize the particle location within the Excel file.
def _run_particle_randomization(workbook: Path, *, persistent: bool) -> bool:
    """
    Randomize particle centres inside the gummy bear mesh using a
    reproducible seed and write the resulting coordinates to the workbook.

    The workbook is displayed before and after randomization to make the
    changes traceable during notebook execution.

    Returns
    -------
    bool
        True when centres already matched the seed (idempotent / unchanged).
    """
    before = pd.read_excel(workbook, sheet_name="particles", engine="openpyxl")[_cols].copy()
    # package functionality: use the mesh to draw the particle positions from a random uniform distribution within the mesh
    result = randomize_workbook_particle_centers(
        workbook,
        seed=_PARTICLE_RANDOM_SEED,
        root=ROOT,
    )
    after = pd.read_excel(workbook, sheet_name="particles", engine="openpyxl")[_cols].copy()
    before_xyz = before[["center_x", "center_y", "center_z"]].to_numpy(dtype=float)
    idempotent = bool(
        np.allclose(before_xyz, result.centers, rtol=1e-12, atol=1e-9)
    )

    print(
        f"Particle randomization: mode={DATA_MODE!r}, "
        f"persistent={persistent}, "
        f"n={result.n_particles}, seed={result.seed}, "
        f"workbook={display_path(workbook)}"
    )
    if idempotent:
        print(
            "Idempotent: centres already matched seed "
            f"{_PARTICLE_RANDOM_SEED} — workbook unchanged."
        )
    else:
        print(
            f"Updated centres with seed={_PARTICLE_RANDOM_SEED} "
            f"({'persistent live workbook' if persistent else 'temp copy only'})."
        )
    print("Before:")
    display(before.head(11))
    print("After:")
    display(after.head(11))
    return idempotent


def _demonstrate_particle_randomization_on_temp_copy(*, reason: str) -> None:
    """Show particle randomization without mutating the catalog workbook."""
    with tempfile.TemporaryDirectory(prefix="gummybear_m8_randomize_") as _tmpdir:
        _tmp_wb = Path(_tmpdir) / Path(WORKBOOK_PATH).name
        shutil.copy2(WORKBOOK_PATH, _tmp_wb)
        _run_particle_randomization(_tmp_wb, persistent=False)
    print(reason)
    print(f"Catalog workbook unchanged: {display_path(WORKBOOK_PATH)}")


# Existing simulations: never rewrite particle centres on the live workbook
# (labels would silently disagree with on-disk images). Inspect always demos on temp.
if DATA_MODE == "inspect" or SIMULATIONS_EXIST:
    if SIMULATIONS_EXIST and DATA_MODE != "inspect":
        print(
            "Existing simulations under OUTPUT_ROOT: particle randomization "
            "is demonstrated on a temporary workbook copy only so particle "
            "labels stay aligned with on-disk images."
        )
    _demonstrate_particle_randomization_on_temp_copy(
        reason=(
            "Temp randomization copy discarded; live/catalog workbook not updated."
        )
    )
else:
    # demo → configs/m8/m8_demo.xlsx ; full → localization_single_particle.xlsx
    _run_particle_randomization(Path(WORKBOOK_PATH), persistent=True)
    print(f"Live workbook: {display_path(WORKBOOK_PATH)}")


#### Train / validation / test split randomization (M8)

Train, validation, and test assignments are generated programmatically (code below) at the particle-identity level and recorded for traceability in the Excel workbook. All observations derived from the same particle configuration inherit the same assignment.

A 70%/15%/15% split was selected as a commonly used compromise between training-set size and independent evaluation subsets. For the demonstration configuration, which contains only 10 particle sequences, a 60%/20%/20% split is used to ensure that each partition contains at least two sequences. This facilitates testing and demonstration of the full processing pipeline while preserving separate training, validation, and test subsets.

Note that fixed train/validation/test randomization is a deliberate project choice. It is trivial to re-randomize the train/val/test split after particle generation (by running the cell below with a different seed and optionally different proportions); however a single predefined split is retained throughout the study to ensure that all architectural variants are evaluated under identical conditions (same rationale as for particle positions, above). Multiple split realizations could be explored in a future robustness study but are outside the scope of the present investigation.

If simulations already exist under the mode output root, re-running this cell still updates the live workbook (unlike particle-centre randomization), but prints a warning: changing the split changes which samples train vs evaluate and therefore the exact ML metrics, even when the images themselves are unchanged.


In [ ]:
_SPLIT_RANDOM_SEED = 53  # stored in sequences.seed
_split_cols = ["sequence_id", "particle_setup_id", "split", "seed"]

# The default fractions are train = 70%, validation = 15%, and test=15%
SPLIT_FRACTIONS = DEFAULT_SPLIT_FRACTIONS

# In the demo mode, the total number of sequences is very small (10) and we wish to have at
# least 2 per group to be able to test variance calculation machinery
if DATA_MODE == "demo":
    SPLIT_FRACTIONS["train"] = 0.6
    SPLIT_FRACTIONS["validation"] = 0.2
    SPLIT_FRACTIONS["test"]=0.2


def _run_split_randomization(workbook: Path, *, persistent: bool) -> bool:
    """
        Randomize particles to train/val/test split. The particle is the unit of randomization;
        same particle is prevented from leaking into different splits.
    
        The workbook is displayed before and after randomization to make the
        changes traceable during notebook execution.

        Returns
        -------
        bool
            True when splits already matched the seed (idempotent / unchanged).
        """
    before = pd.read_excel(workbook, sheet_name="sequences", engine="openpyxl")[
        _split_cols
    ].copy()
    result = randomize_workbook_sequence_splits(
        workbook,
        seed=_SPLIT_RANDOM_SEED,
        train_fraction=SPLIT_FRACTIONS["train"],
        validation_fraction=SPLIT_FRACTIONS["validation"],
        test_fraction=SPLIT_FRACTIONS["test"],
    )
    after = pd.read_excel(workbook, sheet_name="sequences", engine="openpyxl")[
        _split_cols
    ].copy()
    idempotent = bool(
        before["split"].astype(str).equals(after["split"].astype(str))
        and before["seed"].astype(int).equals(after["seed"].astype(int))
        and after["seed"].astype(int).eq(_SPLIT_RANDOM_SEED).all()
    )
    # Same particle never spans multiple splits.
    assert after.groupby("particle_setup_id")["split"].nunique().max() == 1

    print(
        f"Split randomization: mode={DATA_MODE!r}, "
        f"persistent={persistent}, "
        f"n_sequences={result.n_sequences}, n_particles={result.n_particles}, "
        f"seed={result.seed}, workbook={display_path(workbook)}"
    )
    print(f"sequence split_counts={result.split_counts}")
    particle_counts = {name: 0 for name in ("train", "validation", "test")}
    for split_name in result.particle_splits.values():
        particle_counts[split_name] += 1
    print(f"particle split_counts={particle_counts}")
    if idempotent:
        print(
            "Idempotent: splits already matched seed "
            f"{_SPLIT_RANDOM_SEED} — workbook unchanged."
        )
    else:
        print(
            f"Updated splits with seed={_SPLIT_RANDOM_SEED} "
            f"({'persistent live workbook' if persistent else 'temp copy only'})."
        )
    print("Before:")
    display(before.head(11))
    print("After:")
    display(after.head(11))
    return idempotent

# Inspect: demo only on temp. Demo/full: allow live updates; warn if sims exist.
if DATA_MODE == "inspect":
    with tempfile.TemporaryDirectory(prefix="gummybear_m8_split_") as _tmpdir:
        _tmp_wb = Path(_tmpdir) / Path(WORKBOOK_PATH).name
        shutil.copy2(WORKBOOK_PATH, _tmp_wb)
        _run_split_randomization(_tmp_wb, persistent=False)
    print(
        "Temp split-randomization copy discarded; "
        f"catalog workbook unchanged: {display_path(WORKBOOK_PATH)}"
    )
else:
    _split_idempotent = _run_split_randomization(Path(WORKBOOK_PATH), persistent=True)
    print(f"Live workbook ready for generation: {display_path(WORKBOOK_PATH)}")
    if SIMULATIONS_EXIST and not _split_idempotent:
        print(
            "WARNING: Existing simulations under OUTPUT_ROOT, but the "
            "train/validation/test split assignment was updated in the workbook. "
            "Images are unchanged; re-partitioning will change which samples "
            "enter train vs validation/test and therefore the exact ML metrics. "
            "Split/seed are not part of resolved_job_hash, so catalog "
            "completeness is unchanged when only the partition changes."
        )
    elif SIMULATIONS_EXIST:
        print(
            "Note: simulations already exist under OUTPUT_ROOT; splits were "
            "already aligned with the current seed (idempotent)."
        )

if DATA_MODE == "full":
    print("Note: The full workbook contains 3 sequences per particle with different gummybear properties")


#### Step 2: Run optical pipeline (M8)

In [ ]:
generation_result = None
if RUN_GENERATION:
    if not WORKBOOK_PATH.is_file():
        raise FileNotFoundError(f"Workbook not found: {display_path(WORKBOOK_PATH)}")
    generation_result = run_generation_workbook(
        WORKBOOK_PATH,
        repo_root=ROOT,
        stl_root=ROOT,
        output_root=OUTPUT_ROOT,
        cache_root=CACHE_ROOT,
        dry_run=False,
        use_persistent_cache=True,
        remove_stale=True,
        parallel=True,
        progress=True,
    )
    print(generation_result)
else:
    print(
        f"Skipping workbook generation (DATA_MODE={DATA_MODE!r}). "
        "Existing on-disk sequences under OUTPUT_ROOT will be catalogued if present."
    )


#### Step 3: Load catalog from disk (M8)

In [ ]:
# Get the simulation jobs from disk for the current workbook
# catalog_jobs is a list of SequenceJob objects, with nested parameter fields as described below.
# For ease of discussion, consider one entry from this list:
# job = catalog_jobs[i]
# Each job is a SequenceJob object with the following structure:
# - Flat fields such as job.sequence_id, job.split, job.stl_path and others describing this sequence, including cache keys.
# - And fields which are themselves objects for deeper configuration:
#    - job.optical: Optical constants and illumination, of class OpticalSetupConfig
#    - job.particle: Single particle mode only in dataset generation pipeline, of class ParticleSetupConfig.
#    - job.diffusion:  constants and simulation mode, of class DiffusionSetupConfig
#    - job.camera: different camera positions giving rise to the set of views simulated for this job.
#         The job.camera field is of type CameraScheduleConfig, containing common camera information 
#         and a list of CameraPose objects, one for each view (i.e. job.camera.poses) 
#    - There is also a job.corruption field for future work (camera speckles, out of scope in this project)
# 
# Note that the catalog is intentionally derived from the validated SequenceJob representation 
# rather than being loaded independently from the workbook, ensuring consistency between simulation and ML dataset. 
catalog_jobs = load_catalog_jobs(
    WORKBOOK_PATH,
    root_path=ROOT,
    stl_root=ROOT,
)
# Fix: we need to keep job.output_root aligned with the mode's OUTPUT_ROOT as there are different modes (inspect, demo, full)
_output_root_rel = str(repo_relative_path(OUTPUT_ROOT))
catalog_jobs = [
    replace(job, output_root=_output_root_rel) for job in catalog_jobs
]

# Dataset generation and PyTorch data loading operate on a flat, indexable representation.
# Therefore the nested SequenceJob objects are converted into a flattened row structure.
catalog_rows = build_catalog_rows(catalog_jobs)

# And for display, a pandas dataframe is even better.
jobs_df = catalog_jobs_to_dataframe(catalog_jobs)

# Evaluate whether simulations are already completed on disk
status_counts = (
    pd.Series([row.field_status for row in catalog_rows]).value_counts().to_dict()
)
n_complete = sum(1 for row in catalog_rows if row.field_status == "complete")

print(f"catalog jobs: {len(catalog_jobs)}")
print(f"complete on disk: {n_complete}/{len(catalog_rows)}")
print(f"field_status counts: {status_counts}")

_non_complete = [
    {
        "sequence_id": row.sequence_id,
        "split": row.split,
        "field_status": row.field_status,
    }
    for row in catalog_rows
    if row.field_status != "complete"
]
if _non_complete:
    print("non-complete catalog rows (listed, excluded from training filters):")
    display(pd.DataFrame(_non_complete))

# Train / val / test proportions as realized in the catalog (sequence rows).
# This should be identical to the pre-specified proportions
_split_order = ("train", "validation", "test")
_seq_split_counts = (
    jobs_df["split"].astype(str).value_counts().reindex(_split_order, fill_value=0).astype(int)
)
_n_seq = int(_seq_split_counts.sum())
_seq_split_props = {
    name: round((float(count) / _n_seq if _n_seq else 0.0), 3)
    for name, count in _seq_split_counts.items()
}
print(
    f"Train/Val/Test split "
    f"proportions={_seq_split_props}"
)

print(" ")
print("First two rows of catalog:")
# Path columns from the catalog helper are absolute; display repo-relative only.
_path_cols = ("output_root", "sequence_dir", "stl_path", "workbook_path")
_jobs_display = jobs_df.head(2).copy()
for _col in _path_cols:
    if _col in _jobs_display.columns:
        _jobs_display[_col] = _jobs_display[_col].map(display_path)
display(_jobs_display)



#### Step 4: Extract Dataset (M8)

The catalog provides a complete set of information associated with each sequence. The dataset for ML is more restricted:
- In terms of rows, for training, we need the training split, for validation, the validation split, for testing, the testing split.
- In terms of columns, we need to extract:
  - Certain columns as features, also known as "x" or regressors. These features will be used for prediction by the models.
  - Certain columns as labels, also known as "y" or targets. The labels are used to construct the loss function and thus to train the models.

With the idea of a generic optical simulation framework possibly adapted to various tasks beyond this project, I want to have a way of deliberately selecting features, labels and data split from my catalog, while leaving the catalog as loaded for other tasks. To materialize the configuration for a given ML task, I defined a specific class "DatasetTaskSpec" that allows me to select suitable split, features and labels for various ML tasks. I then supply the desired task specification to a helper function build_task_dataset and get the PyTorch-style, indexable dataset that I can use for the ML tasks.

For M8 and M9, the camera views are the features, and the xyz particle positions are the labels, since we want to predict particle position from images in the localization task.

For consistency, images use the common shape [V, C, H, W] in M8 and M9:
   V = number of camera views
   C = image channels (1 for grayscale)

M8 uses single-view acquisition and therefore yields V=1.
The view dimension is retained intentionally so that M8 and M9 share the same dataset interface; V=1 is squeezed as suitable in the later implementation if necessary for feeding the CNN (e.g. Conv2d).

Therefore a sample in M8 is an image represented by a [V=1,C=1,H=128,W=128] tensor, the leading two dimensions being singletons (single view, grayscale). A sample in M9 is an ordered sequence of angular views, and is represented by a [V>1,C=1,H=128,W=128] tensor. Catalog orbits are V=10 in demo and inspection mode, and V=36 in full mode (M9 fusion studies may further subsample by angle stride). The image data is raw float as negative and positive deviations from background are recorded. **Physically, a sample** in M8 and M9 corresponds to the set of camera views acquired for a single particle placed at known xyz position in the gummybear; in M8 one view is used, in M9 all available views are used. In general, the image tensor is provided in the feature data, and the position $x,y,z$ (or in some steps only one coordinate such as $z$) is the label.


In [ ]:
localization_task_M8 = DatasetTaskSpec(
    name="localization_M8",
    # Training split + on-disk identity match (stale_job_hash rows stay in catalog_rows).
    row_filter={"split": "train", "field_status": "complete"},
    # For this project, we use only the particle-attributable signal ("anomaly"), which gives the highest contrast; we want to 
    # answer the scientific question about usefulness of Fourier terms. Optimization of detection of particles from lower contrast
    # images with background signal could be a future extension, but is out of scope now.
    x_fields=("anomaly_ref",),
    # Predict simultaneously the three coordinates of the particle.
    y_fields=("particle_x", "particle_y", "particle_z"),
    keep_angles_deg=0,  # This is the specific switch for M8; keep only one single view but retain the V=1 singleton dimension
)

# Same essential task in M9, retain all the camera views by omitting keep_angles_deg
localization_task_M9 = DatasetTaskSpec(
    name="localization_M9",
    row_filter={"split": "train", "field_status": "complete"},
    x_fields=("anomaly_ref",),
    y_fields=("particle_x", "particle_y", "particle_z"),
)

# Datasets are built as CatalogTaskDataset objects.  
# CatalogTaskDataset is intentionally a lightweight, indexable dataset class
# rather than a subclass of torch.utils.data.Dataset (for portability beyond ML).
#  For the present workflow,
# the required contract is simply:
#
#     x, y = dataset[i]
#
# This keeps the task-dataset layer independent of PyTorch and therefore usable
# beyond deep-learning workflows. If strict PyTorch Dataset inheritance is later needed,
# CatalogTaskDataset could be subclassed without changing the catalog
# or task specification logic.
#
# Also note that CatalogTaskDataset implements lazy loading in __getitem__:
# image data are loaded only when a sample is accessed. This keeps the memory
# footprint low, which is important for image datasets.

dataset_M8 = build_task_dataset(catalog_rows, localization_task_M8)

# Accessor functionality
[x,y] = dataset_M8[0]

print("M8, single view per sample, V=1")
print(f"M8 sample shape={x['anomaly_ref'].shape}")

dataset_M9 = build_task_dataset(catalog_rows, localization_task_M9)

[x,y] = dataset_M9[0]
print(f"\nM9, view set per sample, here V={x['anomaly_ref'].shape[0]} for {360/x['anomaly_ref'].shape[0]}° rotations per camera orbit step")
print(f"M9 sample shape={x['anomaly_ref'].shape}")



#### Sample definition in M8 and M9

Display below: M8 still (single kept camera angle) versus M9 ordered camera orbit for the same particle-localization contract. Tensor shapes follow the `[V,C,H,W]` definition above.


In [ ]:
# Derive display arrays from the task datasets (sample 0), then hand off rendering.

x8, y8 = dataset_M8[0]
x9, y9 = dataset_M9[0]
row8 = dataset_M8.rows[0]
row9 = dataset_M9.rows[0]

# anomaly_ref layout is [V, C, H, W]; M8 keeps V=1, M9 keeps the full orbit.
still_hw = np.asarray(x8["anomaly_ref"], dtype=np.float32)[0, 0]
orbit_vhw = np.asarray(x9["anomaly_ref"], dtype=np.float32)[:, 0]
still_angle_deg = float(localization_task_M8.keep_angles_deg[0])
orbit_angles_deg = tuple(float(a) for a in row9.angles_deg)

display_anomaly_still_vs_orbit(
    still_hw,
    orbit_vhw,
    still_sequence_id=row8.sequence_id,
    orbit_sequence_id=row9.sequence_id,
    still_angle_deg=still_angle_deg,
    orbit_angles_deg=orbit_angles_deg,
    still_heading="M8 single view",
    orbit_heading="M9 multiple views",
    still_caption=(
        f"M8 {tuple(np.asarray(x8['anomaly_ref']).shape)}; "
        "V=1 (still), C=1 (greyscale)"
    ),
    orbit_caption=(
        f"M9 {tuple(np.asarray(x9['anomaly_ref']).shape)}; "
        f"V={orbit_vhw.shape[0]} (orbit), C=1 (greyscale)"
    ),
)

print(
    f"M8 still: {row8.sequence_id}  shape={tuple(np.asarray(x8['anomaly_ref']).shape)}  "
    f"xyz=({y8['particle_x']:.3g}, {y8['particle_y']:.3g}, {y8['particle_z']:.3g})"
)
print(
    f"M9 orbit: {row9.sequence_id}  shape={tuple(np.asarray(x9['anomaly_ref']).shape)}  "
    f"xyz=({y9['particle_x']:.3g}, {y9['particle_y']:.3g}, {y9['particle_z']:.3g})"
)

if DATA_MODE=="inspect":
    print("!!!!!  Images packaged, only assembled and displayed here !!!!!!!!!")


### M10 - Multi-illumination Dataset

M10 extends the M8/M9 sample geometry with a revolving point light. On disk, each sequence is still one (particle, illumination) pair with a camera orbit. For ML, the joint unit uses the canonical **illumination-major** layout:

```text
[I, V, C, H, W]
I = illuminations (lights)
V = camera views
C = channels (1 = greyscale)
H, W = image height, width
```

This matches notebook `10_2` joint units. **10_1** subsamples a fixed camera → `[I, C, H, W]`.

The demo workbook is **hand-maintained** (`tomography_ml_validation` packaged resource / `configs/m10/m10_demo.xlsx`). Current demo scale: **3 illuminations** × **N camera views** (from the Excel camera schedule) × **8 particles** with particle-level **4 / 2 / 2** → **24 sequences**.

#### Step 1: Configuration including particle position and train/val/test randomization


In [ ]:
# M10 path definitions ==================================================================

DEMO_OUTPUT_ROOT = ROOT / "data" / "generated" / "m10_demo"
FULL_OUTPUT_ROOT = ROOT / "data" / "generated" / "m10_illumination"
FULL_WORKBOOK_PATH = ROOT / "configs" / "m10" / "localization_m10_illumination.xlsx"

# Hand-maintained Excel — edit this packaged resource to change demo geometry.
DEMO_PACKAGE_RESOURCE = (
    files("tomography_ml_validation")
    / "test_data"
    / "configs"
    / "m10"
    / "m10_demo.xlsx"
)

def _copy_m10_demo_workbook(dest: Path) -> Path:
    """Copy the packaged M10 demo workbook to ``dest`` with demo ``output_root``."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    with as_file(DEMO_PACKAGE_RESOURCE) as src:
        shutil.copy2(src, dest)
    frames = pd.read_excel(dest, sheet_name=None, engine="openpyxl")
    frames["sequences"]["output_root"] = "data/generated/m10_demo"
    with pd.ExcelWriter(dest, engine="openpyxl") as writer:
        for sheet_name, frame in frames.items():
            frame.to_excel(writer, sheet_name=sheet_name, index=False)
    return dest.resolve()


def _demo_light_angles_from_workbook(path: Path) -> tuple[float, ...]:
    optical = pd.read_excel(path, sheet_name="optical_setups", engine="openpyxl")
    if "light_angle_deg" in optical.columns:
        return tuple(sorted(float(a) for a in optical["light_angle_deg"].tolist()))
    return tuple(
        sorted(
            float(light_angle_deg_from_optical_setup_id(str(oid)))
            for oid in optical["optical_setup_id"].tolist()
        )
    )


if DATA_MODE == "inspect":
    OUTPUT_ROOT = DEMO_OUTPUT_ROOT
    RUN_GENERATION = False
    _inspect_tmpdir = tempfile.TemporaryDirectory(prefix="gummybear_m10_inspect_")
    _SESSION_TEMPDIRS.append(_inspect_tmpdir)
    WORKBOOK_PATH = _copy_m10_demo_workbook(Path(_inspect_tmpdir.name) / "m10_demo.xlsx")
    LIGHT_ANGLES_DEG = _demo_light_angles_from_workbook(WORKBOOK_PATH)
elif DATA_MODE == "demo":
    OUTPUT_ROOT = DEMO_OUTPUT_ROOT
    RUN_GENERATION = True
    WORKBOOK_PATH = _copy_m10_demo_workbook(ROOT / "configs" / "m10" / "m10_demo.xlsx")
    LIGHT_ANGLES_DEG = _demo_light_angles_from_workbook(WORKBOOK_PATH)
else:
    OUTPUT_ROOT = FULL_OUTPUT_ROOT
    RUN_GENERATION = True
    WORKBOOK_PATH = FULL_WORKBOOK_PATH
    LIGHT_ANGLES_DEG = M10_LIGHT_ANGLES_DEG

CACHE_ROOT = OUTPUT_ROOT / "_cache"

SIMULATIONS_EXIST = _output_root_has_simulations(OUTPUT_ROOT)
# Camera angles come from the workbook schedule via the catalog (not hardcoded).
CAMERA_ANGLES_DEG = None
FIXED_CAMERA_DEG = 0.0
FIXED_LIGHT_OPTICAL_ID = f"opt_m10_illum_{int(round(float(LIGHT_ANGLES_DEG[0]))):03d}"

print(f"DATA_MODE={DATA_MODE}")
print(f"RUN_GENERATION={RUN_GENERATION}")
print(f"WORKBOOK_PATH={display_path(WORKBOOK_PATH)}")
print(f"OUTPUT_ROOT={display_path(OUTPUT_ROOT)}")
print(f"CACHE_ROOT={display_path(CACHE_ROOT)}")
print(f"SIMULATIONS_EXIST={SIMULATIONS_EXIST}")
print(f"LIGHT_ANGLES_DEG={LIGHT_ANGLES_DEG}")
print("CAMERA_ANGLES_DEG=None (resolved from workbook/catalog per sample)")
print(f"FIXED_LIGHT_OPTICAL_ID={FIXED_LIGHT_OPTICAL_ID}")


#### Randomization of particle position (M10)

When simulations already exist, the notebook uses a temporary copy to demonstrate randomization. Re-randomizing the Excel worksheet when simulations exist will invalidate the simulations. This would trigger a hash-mismatch error later.


In [ ]:
_PARTICLE_RANDOM_SEED = 43  # The Hitchhiker's Guide to the Galaxy + 1
_cols = ["particle_setup_id", "center_x", "center_y", "center_z", "seed"]



if DATA_MODE == "inspect" or SIMULATIONS_EXIST:
    if SIMULATIONS_EXIST and DATA_MODE != "inspect":
        print(
            "Existing simulations under OUTPUT_ROOT: particle randomization "
            "is demonstrated on a temporary workbook copy only so particle "
            "labels stay aligned with on-disk images."
        )
    with tempfile.TemporaryDirectory(prefix="gummybear_m10_randomize_") as _tmpdir:
        _tmp_wb = Path(_tmpdir) / Path(WORKBOOK_PATH).name
        shutil.copy2(WORKBOOK_PATH, _tmp_wb)
        _run_particle_randomization(_tmp_wb, persistent=False)
    print(
        "Temp randomization copy discarded; "
        f"catalog workbook unchanged: {display_path(WORKBOOK_PATH)}"
    )
else:
    # demo → configs/m10/m10_demo.xlsx ; full → localization_m10_illumination.xlsx
    _run_particle_randomization(Path(WORKBOOK_PATH), persistent=True)
    print(f"Live workbook: {display_path(WORKBOOK_PATH)}")


#### Train / validation / test split randomization (M10)

Reuses the M8 helper ``_run_split_randomization`` (seed **53**, particle-level stratification via ``randomize_workbook_sequence_splits``). All sequences that share a ``particle_setup_id`` (the multi-light joint unit) keep one split.

Default fractions are **70/15/15**. For the small M10 demo (8 particles), fractions are set to **50/25/25** so each partition has at least two particles (4/2/2).

If simulations already exist, split re-randomization is still allowed on the live workbook, with a warning that changing the assignment changes exact ML metrics even when images are unchanged.


In [ ]:
_SPLIT_RANDOM_SEED = 54  # stored in sequences.seed
_split_cols = ["sequence_id", "particle_setup_id", "split", "seed"]


# M10-specific fractions (do not inherit M8 demo 60/20/20 blindly).
SPLIT_FRACTIONS = dict(DEFAULT_SPLIT_FRACTIONS)
if DATA_MODE == "demo":
    # 8 particles → 4 / 2 / 2
    SPLIT_FRACTIONS["train"] = 0.5
    SPLIT_FRACTIONS["validation"] = 0.25
    SPLIT_FRACTIONS["test"] = 0.25

if DATA_MODE == "inspect":
    with tempfile.TemporaryDirectory(prefix="gummybear_m10_split_") as _tmpdir:
        _tmp_wb = Path(_tmpdir) / Path(WORKBOOK_PATH).name
        shutil.copy2(WORKBOOK_PATH, _tmp_wb)
        _run_split_randomization(_tmp_wb, persistent=False)
    print(
        "Temp split-randomization copy discarded; "
        f"catalog workbook unchanged: {display_path(WORKBOOK_PATH)}"
    )
else:
    _split_idempotent = _run_split_randomization(Path(WORKBOOK_PATH), persistent=True)
    print(f"Live workbook ready for generation: {display_path(WORKBOOK_PATH)}")
    if SIMULATIONS_EXIST and not _split_idempotent:
        print(
            "WARNING: Existing simulations under OUTPUT_ROOT, but the "
            "train/validation/test split assignment was updated in the workbook. "
            "Images are unchanged; re-partitioning will change which samples "
            "enter train vs validation/test and therefore the exact ML metrics. "
            "Split/seed are not part of resolved_job_hash, so catalog "
            "completeness is unchanged when only the partition changes."
        )
    elif SIMULATIONS_EXIST:
        print(
            "Note: simulations already exist under OUTPUT_ROOT; splits were "
            "already aligned with the current seed (idempotent)."
        )

if DATA_MODE == "full":
    print(
        "Note: full M10 workbook repeats each particle under multiple illuminations; "
        "splits remain particle-stratified so joint light groups stay in one split."
    )


#### Step 2: Run optical pipeline (M10)

Demo generation writes 24 sequences under `data/generated/m10_demo`. Full mode uses the live M10 illumination workbook (1500 sequences).


In [ ]:
generation_result = None
if RUN_GENERATION:
    if not WORKBOOK_PATH.is_file():
        raise FileNotFoundError(f"Workbook not found: {display_path(WORKBOOK_PATH)}")
    generation_result = run_generation_workbook(
        WORKBOOK_PATH,
        repo_root=ROOT,
        stl_root=ROOT,
        output_root=OUTPUT_ROOT,
        cache_root=CACHE_ROOT,
        dry_run=False,
        use_persistent_cache=True,
        remove_stale=True,
        parallel=True,
        progress=True,
    )
    print(generation_result)
else:
    print(
        f"Skipping workbook generation (DATA_MODE={DATA_MODE!r}). "
        "Existing on-disk sequences under OUTPUT_ROOT will be catalogued if present."
    )


#### Step 3: Load catalog from disk (M10)


In [ ]:
catalog_jobs = load_catalog_jobs(
    WORKBOOK_PATH,
    root_path=ROOT,
    stl_root=ROOT,
)
_output_root_rel = str(repo_relative_path(OUTPUT_ROOT))
catalog_jobs = [replace(job, output_root=_output_root_rel) for job in catalog_jobs]
catalog_rows = build_catalog_rows(catalog_jobs)
jobs_df = catalog_jobs_to_dataframe(catalog_jobs)

status_counts = (
    pd.Series([row.field_status for row in catalog_rows]).value_counts().to_dict()
)
n_complete = sum(1 for row in catalog_rows if row.field_status == "complete")

print(f"catalog jobs: {len(catalog_jobs)}")
print(f"complete on disk: {n_complete}/{len(catalog_rows)}")
print(f"field_status counts: {status_counts}")

_non_complete = [
    {
        "sequence_id": row.sequence_id,
        "split": row.split,
        "field_status": row.field_status,
    }
    for row in catalog_rows
    if row.field_status != "complete"
]
if _non_complete:
    print("non-complete catalog rows (listed, excluded from training filters):")
    display(pd.DataFrame(_non_complete))

_split_order = ("train", "validation", "test")
_seq_split_counts = (
    jobs_df["split"].astype(str).value_counts().reindex(_split_order, fill_value=0).astype(int)
)
_n_seq = int(_seq_split_counts.sum())
_seq_split_props = {
    name: round((float(count) / _n_seq if _n_seq else 0.0), 3)
    for name, count in _seq_split_counts.items()
}
print(f"Train/Val/Test split (sequence rows) proportions={_seq_split_props}")

_path_cols = ("output_root", "sequence_dir", "stl_path", "workbook_path")
_jobs_display = jobs_df.head(2).copy()
for _col in _path_cols:
    if _col in _jobs_display.columns:
        _jobs_display[_col] = _jobs_display[_col].map(display_path)
print(" ")
print("First two rows of catalog:")
display(_jobs_display)


#### Step 4: Prepare Dataset (M10)

M8/M9 use a 4D view stack `[V, C, H, W]`. M10 joint samples use a **5D illumination-major grid**:

```text
views[light_i, cam_j]  →  shape [I, V, C, H, W]
light_angles [I]
camera_angles [V]
```

One ML sample is one particle observed under the full light × camera set, with shared xyz labels. On disk this is assembled from one catalog sequence per light (each sequence already carries the camera orbit). **10_1** keeps a single camera index (`V=1` after subsample).


In [ ]:
joint_groups = build_illumination_joint_groups(
    catalog_rows,
    light_angles_deg=LIGHT_ANGLES_DEG,
    complete_only=True,
    min_groups=0,
)
print(f"joint illumination groups: {count_groups_by_split(joint_groups)}")

train_groups = groups_for_split(joint_groups, "train")
if not train_groups:
    raise RuntimeError(
        "No complete train multi-light groups on disk. "
        "Set DATA_MODE='demo' (or 'full') and run generation, or place "
        "existing sequences under OUTPUT_ROOT."
    )

# Demo / inspect: use workbook camera angles. Full: keep every orbit angle.
if CAMERA_ANGLES_DEG is not None:
    camera_angles_for_sample = tuple(float(a) for a in CAMERA_ANGLES_DEG)
else:
    _ref_row = train_groups[0]["rows_by_light"][float(LIGHT_ANGLES_DEG[0])]
    camera_angles_for_sample = tuple(float(a) for a in _ref_row.angles_deg)

X_FIELD_M10 = "anomaly_ref"
dataset_M10 = HierarchicalCameraLightDataset(
    train_groups,
    x_field=X_FIELD_M10,
    y_fields=("particle_x", "particle_y", "particle_z"),
    camera_angles_deg=camera_angles_for_sample,
    light_angles_deg=LIGHT_ANGLES_DEG,
    image_normalize="none",
    task_name="localization_M10_hierarchical",
    row_filter={"field_status": "complete"},
)

x_m10, y_m10 = dataset_M10[0]
views = x_m10[X_FIELD_M10]
print(
    f"M10 joint sample (canonical): shape={tuple(views.shape)}  "
    f"[I, V, C, H, W]"
)
print(
    f"lights={dataset_M10.light_angles_deg}  "
    f"cameras={dataset_M10.camera_angles_deg}"
)
print(
    f"xyz=({y_m10['particle_x']:.3g}, {y_m10['particle_y']:.3g}, {y_m10['particle_z']:.3g})"
)


#### Sample definition in M10

For ML, the joint unit uses the canonical **illumination-major** layout `[I, V, C, H, W]` (see above). A single GIF sweeps that light × camera grid in the same ordering as the 5D sample tensor: all camera views per light, then the next light. GIF rate is **2 fps** in `inspect` / `demo` and **10 fps** in `full`.


In [ ]:
x_m10, y_m10 = dataset_M10[0]
group0 = dataset_M10.groups[0]
particle_id = group0["particle_id"]
seq_id = group0["rows_by_light"][float(LIGHT_ANGLES_DEG[0])].sequence_id

grid = np.asarray(x_m10[X_FIELD_M10], dtype=np.float32)  # [I, V, C, H, W]
cam_angles_deg = dataset_M10.camera_angles_deg
light_angles_deg = dataset_M10.light_angles_deg

_m10_gif_fps = 10.0 if DATA_MODE == "full" else 2.0
_n_light, _n_cam = int(grid.shape[0]), int(grid.shape[1])
display_anomaly_camera_light_grid(
    None,
    grid,
    sequence_id=seq_id,
    camera_angles_deg=cam_angles_deg,
    light_angles_deg=light_angles_deg,
    fps=_m10_gif_fps,
    include_still=False,
    heading="M10 multi-view, multi-illumination",
    caption=(
        f"M10 sample {tuple(grid.shape)} = [I,V,C,H,W]  "
        f"GIF sweeps {_n_light * _n_cam} frames @ {_m10_gif_fps:g} fps"
    ),
)

print(
    f"M10 sample: particle={particle_id}  sequence={seq_id}  "
    f"shape={tuple(grid.shape)}  "
    f"xyz=({y_m10['particle_x']:.3g}, {y_m10['particle_y']:.3g}, {y_m10['particle_z']:.3g})"
)


if DATA_MODE=="inspect":
    print("!!!!!  Images packaged, only assembled and displayed here !!!!!!!!!")


### Dataset Validity



#### Dataset gathering

The dataset is fully synthetic and generated through a custom optical tomography simulation pipeline. For each sample, particle positions and optical simulation parameters are drawn from predefined random seeds, optical propagation is simulated, and multi-view images are generated. The complete generation process is therefore reproducible from the stored configuration and deterministic in terms of known, registered random seeds for particle positions and train/val/test split.

#### Statistical validity

Statistical independence between train, validation, and test partitions is enforced at the particle level. Samples are assigned to a split using a dedicated split randomization seed, while particle placement uses separate randomization seeds. This separation ensures that train/validation/test assignment is independent of particle generation. The per-particle assignment prevents data leakage between partitions including in multi-view and multi-illumination settings.

Train/validation/test assignments are generated once during dataset definition, recorded in the dataset configuration, and remain fixed throughout all experiments. Consequently, model evaluation is always performed on a predefined partitioning established before any training or analysis is conducted. This avoids inadvertent bias from repeated split selection and ensures that all model comparisons are evaluated on identical data partitions.



#### Data Cleaning

No generated samples were discarded during dataset creation. This includes difficult or low-information cases, such as configurations in which particle visibility is reduced (or in rare cases, null) by the optical geometry. Retaining all generated samples avoids introducing selection bias and provides a more representative evaluation of model performance.

No manual cleaning or sample selection was performed.

#### Data manipulation
Generated images are transformed into ML-ready tensors as outlined above. Prior to entering the machine learning pipeline, they are normalized using per-view z-score normalization (see below).

#### Data integrity

To ensure consistency between image data and associated metadata, a hash derived from the generation conditions is compared against a corresponding hash stored in the sequence manifest. This validation step helps detect accidental mismatches between generated images and their labels before dataset construction.

# 5. Deep learning


## Core Network architecture

The aim of this project is to evaluate utility of Fourier-type modulation of encodings for spatial localization from camera views of particle signal in a translucent gummybear phantom.

This section shows the bare-minimum prediction pass in direct PyTorch. Packaged variants used in the M8–M10 studies live in `tomography_ml`. The canonical network is:

- 3-layer CNN image trunk
- Fourier pooling
- Small MLP projecting to predicted coordinates

The loss function used throughout this project is mean squared error (MSE), reflecting Euclidean geometry when localizing particles in 3D space. Parameter count is discussed as an additional measure with possible mobile or embedded deployment in mind.

Below: a hard-coded forward pass on one M8 training sample (untrained weights), including optimizer setup (Adam) and an example training step.


In [ ]:
# imports ================================================================
import torch

from tomography_ml.localization.fourier_pool import enumerate_fourier_modes
from tomography_ml_validation import ensure_m8_illustration_dataset
from torch.nn.functional import mse_loss

# Feature and Label Tensor preparation ===================================

dataset_M8, localization_task_M8 = ensure_m8_illustration_dataset(
    ROOT, DATA_MODE, globals()
)

# Use the first image in the M8 training set for illustration.
x0, y0 = dataset_M8[0]

# Extract the view stack. M8/M9 samples are stored as a set of views.
# For M8, V=1, so the shape is (V, C_in, H, W) = (1, C_in, H, W).
views = x0[localization_task_M8.x_fields[0]]

# Convert to a PyTorch float tensor. The anomaly images are continuous-valued.
views = torch.as_tensor(views, dtype=torch.float32)

# Remove the view dimension to obtain one image: (C_in, H, W).
# Then add a batch dimension because Conv2d expects batched input: (B, C_in, H, W).
img = views.squeeze(0)
x = img.unsqueeze(0)

#We express the label similarly as tensor and unsqueeze it for B=1
y = torch.tensor(
    [y0["particle_x"], y0["particle_y"], y0["particle_z"]],
    dtype=torch.float32,
).unsqueeze(0)

print("input  [B, C_in, H, W] =", tuple(x.shape))
print("target particle pos    =", (y0["particle_x"], y0["particle_y"], y0["particle_z"]))

# CNN network definition ================================================

# 3-layer CNN trunk encoder.
# The number of feature channels increases with depth.
c0, c1, c2 = 16, 32, 64

# First layer: the number of input channels is dictated by the image,
# here C_in=1 for grayscale anomaly images.
# kernel_size=3 with padding=1 preserves the spatial dimensions H and W.
conv1 = torch.nn.Conv2d(x.shape[1], c0, kernel_size=3, padding=1)

# The next layers use the previous layer's output channel count as input channel count.
conv2 = torch.nn.Conv2d(c0, c1, kernel_size=3, padding=1)
conv3 = torch.nn.Conv2d(c1, c2, kernel_size=3, padding=1)

# CNN network application =============================================

# Conv2d is a torch.nn.Module and implements __call__, so it can be used like a function.
# ReLU adds the non-linearity after each convolution.
h = torch.relu(conv1(x))
print("conv1  [B, 16, H, W]   =", tuple(h.shape))

h = torch.relu(conv2(h))
print("conv2  [B, 32, H, W]   =", tuple(h.shape))

feat = torch.relu(conv3(h))
print("feat   [B, 64, H, W]   =", tuple(feat.shape))

B, C, H, W = feat.shape

# Fourier head: Definition =============================================================

# Fourier pooling.
# Assign one low-frequency Fourier mode to each CNN feature channel.
# The mode sequence starts with the constant mode and then proceeds through
# sine/cosine modes of increasing spatial frequency. Use a helper to get these modes
modes = enumerate_fourier_modes(C)

# Show only the first five modes for readability.
print(f"Fourier mode listing: {modes[:5]}")

# Construct coordinate grids over the image domain.
# yy increases down the image rows, xx increases across the image columns.
# Both tensors have shape (H, W).
yy, xx = torch.meshgrid(
    torch.linspace(0.0, 2.0 * math.pi, H),
    torch.linspace(0.0, 2.0 * math.pi, W),
    indexing="ij",
)

# Fourier head: Application of Fourier coefficients ===========================

# The basis tensor stores one Fourier basis map per CNN feature channel.
basis = torch.empty(C, H, W)

# For each mode, construct the phase image kx*x + ky*y and evaluate either
# cos(phase) or sin(phase) to obtain the corresponding Fourier basis map.
for i, (kx, ky, kind) in enumerate(modes):
    if kind == "const":
        basis[i].fill_(1.0)
    else:
        phase = float(kx) * xx + float(ky) * yy
        basis[i] = torch.cos(phase) if kind == "cos" else torch.sin(phase)

print("basis  [C, H, W]       =", tuple(basis.shape), f"  first modes={modes[:4]} ...")

# Fourier-coded pooling ======================================================
# multiply each feature channel with its assigned basis map, then average over H and W.
# basis.unsqueeze(0) adds the batch dimension so broadcasting matches feat: (B, C, H, W).
fourier_pooled = (feat * basis.unsqueeze(0)).mean(dim=(-2, -1))

print("Fourier pooled [B, C]          =", tuple(fourier_pooled.shape))


# Minimalsit MLP head for coordinate prediction (Lin -> ReLU -> Lin) ===========
# n_out matches the localization target definition, for example particle (x,y,z) -> 3.
n_out = len(localization_task_M8.y_fields)
hidden = 128

fc1 = torch.nn.Linear(C, hidden)
fc2 = torch.nn.Linear(hidden, n_out)

pred = fc2(torch.relu(fc1(fourier_pooled)))

print("hidden [B, hidden]     =", (B, hidden))

# The prediction is random here because this miniature network has not been trained.
# Without detach(), PyTorch keeps the autograd graph attached, which can cause
# surprising failures when converting tensors to NumPy or reusing values. Be safe.
print("pred   [B, n_out]      =", tuple(pred.shape), "  value=", pred.detach().cpu().numpy())


# Loss and one optimizer step ===============================================

# Optimizer setup, typically done after model setup but before start of training loop
lr = 0.03  # example learning rate

params = (
    list(conv1.parameters())
    + list(conv2.parameters())
    + list(conv3.parameters())
    + list(fc1.parameters())
    + list(fc2.parameters())
)

opt = torch.optim.Adam(params, lr=float(lr))

# One optimizer step:
# 1. clear old gradients,
# 2. compute loss,
# 3. compute gradients by backpropagation,
# 4. update the trainable parameters.
opt.zero_grad()

# Evaluate the regression loss between predicted and true particle coordinates.
# At this point, the network is randomly initialized, so the prediction is whatever it is.
loss_before = mse_loss(pred, y)

#Be safe: for just output, detach, so it doesn't get counted as connected, back-propagatable operation
print("target [B, n_out]      =", tuple(y.shape), "  value=", y.detach().cpu().numpy())
print("loss before step       =", float(loss_before.detach()))
print("RMSE before step       =", float(torch.sqrt(loss_before.detach())))



loss_before.backward()
opt.step()

# Recompute the forward pass after the optimizer step.
# The old loss tensor does not update automatically when parameters change.
h_after = torch.relu(conv1(x))
h_after = torch.relu(conv2(h_after))
feat_after = torch.relu(conv3(h_after))

fourier_pooled_after = (feat_after * basis.unsqueeze(0)).mean(dim=(-2, -1))

pred_after = fc2(torch.relu(fc1(fourier_pooled_after)))

loss_after = mse_loss(pred_after, y)

print("pred after step [B, n_out] =", tuple(pred_after.shape),
      "  value=", pred_after.detach().cpu().numpy())

print("loss after step        =", float(loss_after.detach()))
print("RMSE after step        =", float(torch.sqrt(loss_after.detach())))



## M8. Milestone M8: Single-view localization studies

Three spatial-readout architectures share the same CNN trunk and differ only in how spatial information is read out:

| Variant | Readout | Path |
|---|---|---|
| **pooled** | zero-order | CNN → global avg pool → Linear → targets |
| **fourier** | fixed low-frequency | CNN → Fourier-coded pool → Linear → targets |
| **flatten** | full learned | CNN → Flatten → MLP → targets |

**Protocol (in order):**

1. **Learning-rate study on `particle_z` only** (geometrically most evident axis) — illustrative sweep over a learning-rate grid; subsequent studies always use the historical consensus LRs (`M8_CANONICAL_LR_BY_ARCH`), regardless of which LR the sweep prefers
2. **Full train → validation / test on `particle_z`.** This directly compares the architectures to each other in the clearest setting (geometrically most distinct axis)
3. **Full train → validation / test on `(particle_x, particle_y, particle_z)`.** Challenge in a richer prediction setting
4. **Split sensitivity on xyz:** repeat step 3 for `N_sensitivity` new particle-level splits (seeds 60–64), with **one** training run per split. Permits to understand the robustness of the result in the face of train/val/test re-randomization. Sensitivity only: Primary split is authoritative.

**Input:** `anomaly_ref` (particle-attributable signal; no noise or artificial image corruption), single view at 180° (`keep_angles_deg=180`). Per-view z-score normalization is applied (including within image series) to maximize contrast: the aim is to assess architecture, not robustness to poor image quality.

**Data mode:** `inspect` / `demo` use the M8 demo corpus with shorter schedules; `full` uses the full M8 localization corpus.

Note that historical runs are registered at `data/generated/m8_1`, for historical reasons with folder names starting with `_08_3a`. In this way, re-run data (different seeds) is accumulated for historical analysis of larger result sets.


### M8 Study setup

In the shared setup, folder locations, the M8 dataset and common model parameters are defined.


In [ ]:
# Set False to skip the illustrative LR sweep. Subsequent studies always use M8_CANONICAL_LR_BY_ARCH.
RUN_LR_STUDY = True
# Split-sensitivity (point 4): one training run per split seed.


import torch

from tomography_ml import get_device
from tomography_ml.studies import (
    M08_TRAIN_VAL_TEST_XYZ,
    M08_TRAIN_VAL_TEST_Z,
    M08_XYZ_SPLIT_SENSITIVITY,
    study_checkpoint_policy,
    study_results_dir,
    ARCH_ORDER,
    M8_CANONICAL_LR_BY_ARCH,
    DEFAULT_LR_GRID,
    DEFAULT_SENSITIVITY_SPLIT_SEEDS,
    probe_m8_parameter_counts,
    run_learning_rate_study,
    run_split_sensitivity_study,
    run_train_val_test_study,
)
from tomography_ml_validation.plotting import (
    apply_report_titles,
    plot_error_histograms,
    plot_learning_rate_study,
    plot_rmse_summary_bars,
)
from tomography_ml_validation.run_history import load_summary_for_plots

# M8 paths (earlier M10 cells in the optis part may overwrite WORKBOOK_PATH / OUTPUT_ROOT).
_M8_DEMO_WORKBOOK = ROOT / "configs" / "m8" / "m8_demo.xlsx"
_M8_FULL_WORKBOOK = ROOT / "configs" / "m8" / "localization_single_particle.xlsx"
_M8_DEMO_DATA_ROOT = ROOT / "data" / "generated" / "m8_demo"
_M8_FULL_DATA_ROOT = ROOT / "data" / "generated" / "m8_1" / "single_particle"

#In ML, there is no pure "inspect" part, we shunt "inspect" into "demo"
# This means that in all cases, at least the demo dataset will run.
# The demo dataset is big enough only to demonstrate running code, deep learning does not yield
# useful results at this scale: val and split are too different from train.
if DATA_MODE in ("inspect", "demo"):
    STUDY_WORKBOOK_PATH = _M8_DEMO_WORKBOOK
    STUDY_DATA_ROOT = _M8_DEMO_DATA_ROOT
    if not STUDY_WORKBOOK_PATH.is_file():
        STUDY_WORKBOOK_PATH = _copy_demo_workbook(STUDY_WORKBOOK_PATH)
    NUM_EPOCHS_LR = 30
    NUM_EPOCHS_FULL = 40
    N_REPEAT_TRAINING = 1
    LR_GRID = (1e-4, 1e-3, 1e-2, 3e-2, 1e-1)
else:
    STUDY_WORKBOOK_PATH = _M8_FULL_WORKBOOK
    STUDY_DATA_ROOT = _M8_FULL_DATA_ROOT
    # Full-corpus defaults (08_3A_0 notebook used up to 1000 LR epochs;
    # 200 keeps the report tractable while retaining the same protocol).
    NUM_EPOCHS_LR = 200
    NUM_EPOCHS_FULL = 200
    N_REPEAT_TRAINING = 3
    LR_GRID = DEFAULT_LR_GRID

# Full-mode checkpoints under checkpoints/m8/; demo/inspect use STUDY_DATA_ROOT sidecars.
from gummybear.paths import checkpoint_dir

_M8_CKPT = study_checkpoint_policy(
    repo_root=ROOT,
    milestone="m8",
    data_mode=DATA_MODE,
    read_checkpoints=READ_CHECKPOINTS,
)
RESULTS_DIR_LR = checkpoint_dir(ROOT, "m8") if DATA_MODE == "full" else None
RESULTS_DIR_Z = study_results_dir(
    _M8_CKPT, fallback=STUDY_DATA_ROOT / "_08_3a2_train_validation_study_z"
)
RESULTS_DIR_XYZ = study_results_dir(
    _M8_CKPT, fallback=STUDY_DATA_ROOT / "_08_3a2_train_validation_study_xyz"
)
RESULTS_DIR_SENS = study_results_dir(
    _M8_CKPT, fallback=STUDY_DATA_ROOT / "_08_3a2_xyz_split_sensitivity"
)

BATCH_SIZE = 32
BASE_SEED = 0

N_SENSITIVITY = 5
SENSITIVITY_SPLIT_SEEDS = DEFAULT_SENSITIVITY_SPLIT_SEEDS[:N_SENSITIVITY]
assert len(SENSITIVITY_SPLIT_SEEDS) == N_SENSITIVITY

device = get_device()
print(f"DATA_MODE={DATA_MODE}")
print(f"READ_CHECKPOINTS={READ_CHECKPOINTS}")
print(f"STUDY_WORKBOOK_PATH={display_path(STUDY_WORKBOOK_PATH)}")
print(f"STUDY_DATA_ROOT={display_path(STUDY_DATA_ROOT)}")
print(f"RESULTS_DIR_LR={display_path(RESULTS_DIR_LR) if RESULTS_DIR_LR else '-'}")
print(f"Using device: {device}")
print(
    f"NUM_EPOCHS_LR={NUM_EPOCHS_LR}  NUM_EPOCHS_FULL={NUM_EPOCHS_FULL}  "
    f"N_REPEAT_TRAINING={N_REPEAT_TRAINING}  RUN_LR_STUDY={RUN_LR_STUDY}  "
    f"N_SENSITIVITY={N_SENSITIVITY} seeds={SENSITIVITY_SPLIT_SEEDS}"
)

study_catalog_jobs = load_catalog_jobs(
    STUDY_WORKBOOK_PATH, STUDY_DATA_ROOT, stl_root=ROOT
)
study_catalog_rows = build_catalog_rows(study_catalog_jobs)

_common_filter = {
    "split": "train",
    "field_status": "complete",
    "optical_setup_id": "opt_m8_high_001",
}

# z-only first (easiest axis), then full xyz.
localization_task_z = DatasetTaskSpec(
    name="localization_z",
    row_filter=dict(_common_filter),
    x_fields=("anomaly_ref",),
    y_fields=("particle_z",),
    keep_angles_deg=180,
    image_normalize="per_image_zscore",
)
localization_task_xyz = DatasetTaskSpec(
    name="localization_xyz",
    row_filter=dict(_common_filter),
    x_fields=("anomaly_ref",),
    y_fields=("particle_x", "particle_y", "particle_z"),
    keep_angles_deg=180,
    image_normalize="per_image_zscore",
)

dataset_study_z = build_task_dataset(study_catalog_rows, localization_task_z)
print(dataset_study_z)
print(
    "param counts (z-only):",
    probe_m8_parameter_counts(
        dataset=dataset_study_z,
        x_field=localization_task_z.x_fields[0],
        n_outputs=len(localization_task_z.y_fields),
        device=device,
    ),
)

def demo_warning(mode: str, fig=None):
    """Stamp a demo-corpus banner on an ML figure (inspect/demo only)."""
    if mode not in ("inspect", "demo") or fig is None:
        return fig
    if getattr(fig, "_gummybear_demo_warning", False):
        return fig
    fig.text(
        0.5,
        0,
        "DEMO DATASET (6/2/2) • EXECUTION TEST ONLY • DO NOT INTERPRET PERFORMANCE NUMBERS",
        ha="center",
        va="bottom",
        fontsize=22,
        color="darkred",
        weight="bold",
    )
    fig._gummybear_demo_warning = True
    return fig


def show_ml(fig=None, *, heading=None, caption=None):
    """Apply report titles (optional) and demo_warning, then plt.show."""
    if fig is not None and heading:
        apply_report_titles(fig, heading, caption)
    demo_warning(DATA_MODE, fig)
    plt.show()


def _ml_plt_show_restore(*, heading=None, caption=None):
    """Patch plt.show to title+banner only newly opened figures; return original show."""
    before = set(plt.get_fignums())
    _show = plt.show

    def _wrapped(*args, **kwargs):
        for num in plt.get_fignums():
            if num not in before:
                fig = plt.figure(num)
                if heading:
                    apply_report_titles(fig, heading, caption)
                demo_warning(DATA_MODE, fig)
        return _show(*args, **kwargs)

    plt.show = _wrapped
    return _show


### M8 Step 1: Learning-rate study

For each architecture, train on the full training split over a learning-rate grid and report the LR with lowest **validation MSE** (thickened in the curves below). This sweep is **illustrative only**: subsequent train/val studies always use `M8_CANONICAL_LR_BY_ARCH` (historical consensus, for reproducibility), not the sweep-selected LRs. Study on foot to head axis, which is geometrically cleanest. No evaluation on test split. In this step, no scientific conclusions are drawn.


In [ ]:
# Subsequent studies always use historical consensus LRs.
LR_BY_ARCH = dict(M8_CANONICAL_LR_BY_ARCH)

if RUN_LR_STUDY:
    # Demo/inspect must not read or write checkpoints/m8 (would clobber full-scale).
    _lr_use_checkpoint = DATA_MODE == "full"
    lr_study = run_learning_rate_study(
        catalog_rows=study_catalog_rows,
        task=localization_task_z,
        device=device,
        lr_grid=LR_GRID,
        num_epochs=NUM_EPOCHS_LR,
        batch_size=BATCH_SIZE,
        seed=BASE_SEED,
        results_dir=RESULTS_DIR_LR if _lr_use_checkpoint else None,
        load_existing=bool(READ_CHECKPOINTS and _lr_use_checkpoint),
        retrain=bool(_lr_use_checkpoint and not READ_CHECKPOINTS),
    )
    study_lr_by_arch = lr_study.lr_by_arch
    print(
        f"LR study skipped_train={lr_study.skipped_train}  "
        f"checkpoint={display_path(lr_study.checkpoint_path)}"
    )
    print("Sweep-preferred LR_BY_ARCH (min validation MSE on z; illustrative only):")
    for arch in ARCH_ORDER:
        best = study_lr_by_arch[arch]
        val = lr_study.lr_results[arch][best]["val_mse"]
        print(f"  {arch:8s}  lr={best:g}  val_MSE={val:.6g}")
    fig = plot_learning_rate_study(
        lr_study.lr_results,
        study_lr_by_arch,
        lr_grid=lr_study.lr_grid,
        num_epochs=lr_study.num_epochs,
        x_field=lr_study.x_field,
        y_fields=lr_study.y_fields,
        train_size=lr_study.train_size,
        val_size=lr_study.val_size,
        heading="M8 Step 1 — learning-rate study",
        caption=(
            f"z-only  |  {lr_study.num_epochs} epochs  |  "
            f"x={lr_study.x_field}  y={list(lr_study.y_fields)}  |  "
            f"train n={lr_study.train_size}  val n={lr_study.val_size}"
        ),
    )
    show_ml(fig)
else:
    print("Skipped LR sweep (illustration only when RUN_LR_STUDY=True).")

print("\n=== LR_BY_ARCH (historical; used by subsequent train/val studies) ===")
for arch in ARCH_ORDER:
    print(f"  {arch:8s}  lr={LR_BY_ARCH[arch]:g}")
print(LR_BY_ARCH)


### M8 Step 2. z-axis localization study. Train → validation / test on `particle_z`

Test of Fourier vs. Pooling layers, with Flattened as full embedding retention control. Restricted to the z-axis. Each architecture is trained on the full training split with the historical canonical LR for that architecture (`M8_CANONICAL_LR_BY_ARCH`), then evaluated on validation and test.

This step answers the question of the performance of the Fourier layer with respect to GAP pooling (spatial averaging) and flatten (full embedding retention) in the simplified task of z-position estimation (foot-head coordinate). The design of the study means that performance evaluation is interpreted for this particular dataset and split; some training variability is taken into account by evaluation over 3 repeated training runs.


In [ ]:
study_z = run_train_val_test_study(
    catalog_rows=study_catalog_rows,
    task=localization_task_z,
    device=device,
    results_dir=RESULTS_DIR_Z,
    lr_by_arch=LR_BY_ARCH,
    notebook_id="08_3A_2",
    experiment_id="m08_3a2_train_validation_study_z",
    variant_prefix="m08_3a2z",
    num_epochs=NUM_EPOCHS_FULL,
    batch_size=BATCH_SIZE,
    n_repeat_training=N_REPEAT_TRAINING,
    base_seed=BASE_SEED,
    csv_run_history="m08_train_val_test_z_run_history.csv",
    csv_session_summary="m08_train_val_test_z_session_summary.csv",
    csv_comparison="m08_train_val_test_z_comparison.csv",
    architecture_note_prefix="M8 z-only",
    checkpoint_name=M08_TRAIN_VAL_TEST_Z if _M8_CKPT.enabled else None,
    load_existing=_M8_CKPT.load_existing,
    retrain=_M8_CKPT.retrain,
)
print(f"study_z skipped_train={study_z.skipped_train}  checkpoint={display_path(study_z.checkpoint_path)}")

print(
    f"train={study_z.train_size}  validation={study_z.val_size}  "
    f"test={study_z.test_size}  n_rep={study_z.n_rep}"
)
print(f"Wrote {repo_relative_path(study_z.history_path)}")
print(f"Wrote {repo_relative_path(study_z.session_summary_path)}")
print(study_z.session_summary_df.to_string(index=False))

summary_z, summary_src_z = load_summary_for_plots(
    study_z.history_path,
    session_run_ids=study_z.session_run_ids,
    session_summary=study_z.session_summary_df,
    session_summary_path=study_z.session_summary_path,
)
print(f"summary source: {summary_src_z}")

fig, stats_z = plot_rmse_summary_bars(summary_z, variant_prefix="m08_3a2z")
show_ml(
    fig,
    heading="M8 Step 2 — Validation/Test particle-z",
    caption=(
        "z-only validation / test RMSE "
        f"(mean ± SD across runs; n={stats_z['n_runs']})"
    ),
)

fig = plot_error_histograms(
    study_z.full_results,
    title=(
        f"z-only error histograms | x={study_z.x_field} y={list(study_z.y_fields)} "
        f"| last of {study_z.n_rep} runs"
    ),
)
show_ml(fig, heading="M8 Step 2 — z-axis error histograms")

print("z-only validation / test RMSE across training runs:")
for arch, vm, vs, tm, ts, n in zip(
    ARCH_ORDER,
    stats_z["val_means"],
    stats_z["val_stds"],
    stats_z["test_means"],
    stats_z["test_stds"],
    stats_z["n_runs_list"],
):
    print(
        f"  {arch:8s}  val mean={vm:.4f}  sd={vs:.4f}  "
        f"test mean={tm:.4f}  sd={ts:.4f}  n_runs={n}"
    )


### M8 Step 3 Train → validation / test on `(x, y, z)`

Repeat the full train → val/test comparison with three-dimensional targets, reusing the historical canonical learning rates. Again, no loss-vs-epoch overlay — only held-out RMSE summaries. Full 3D target leads to higher RMSE losses (three coordinates) and could potentially be more variable.

This step answers the question of whether the performance advantage of the Fourier layer with relative to pooling, approaching flattening, extends to the task of 3D position estimation. Again, the design of the study means that performance evaluation is interpreted for this particular dataset and split; some training variability is taken into account by evaluation over 3 repeated training runs.




In [ ]:
study_xyz = run_train_val_test_study(
    catalog_rows=study_catalog_rows,
    task=localization_task_xyz,
    device=device,
    results_dir=RESULTS_DIR_XYZ,
    lr_by_arch=LR_BY_ARCH,
    notebook_id="08_3A_2",
    experiment_id="m08_3a2_train_validation_study_xyz",
    variant_prefix="m08_3a2xyz",
    num_epochs=NUM_EPOCHS_FULL,
    batch_size=BATCH_SIZE,
    n_repeat_training=N_REPEAT_TRAINING,
    base_seed=BASE_SEED,
    csv_run_history="m08_train_val_test_xyz_run_history.csv",
    csv_session_summary="m08_train_val_test_xyz_session_summary.csv",
    csv_comparison="m08_train_val_test_xyz_comparison.csv",
    architecture_note_prefix="M8 xyz",
    checkpoint_name=M08_TRAIN_VAL_TEST_XYZ if _M8_CKPT.enabled else None,
    load_existing=_M8_CKPT.load_existing,
    retrain=_M8_CKPT.retrain,
)
print(f"study_xyz skipped_train={study_xyz.skipped_train}  checkpoint={display_path(study_xyz.checkpoint_path)}")

print(
    f"train={study_xyz.train_size}  validation={study_xyz.val_size}  "
    f"test={study_xyz.test_size}  n_rep={study_xyz.n_rep}"
)
print(f"Wrote {repo_relative_path(study_xyz.history_path)}")
print(f"Wrote {repo_relative_path(study_xyz.session_summary_path)}")
print(study_xyz.session_summary_df.to_string(index=False))

summary_xyz, summary_src_xyz = load_summary_for_plots(
    study_xyz.history_path,
    session_run_ids=study_xyz.session_run_ids,
    session_summary=study_xyz.session_summary_df,
    session_summary_path=study_xyz.session_summary_path,
)
print(f"summary source: {summary_src_xyz}")

fig, stats_xyz = plot_rmse_summary_bars(summary_xyz, variant_prefix="m08_3a2xyz")
show_ml(
    fig,
    heading="M8 Step 3 — Validation/Test particle x,y,z",
    caption=(
        "xyz validation / test RMSE "
        f"(mean ± SD across runs; n={stats_xyz['n_runs']})"
    ),
)

print("xyz validation / test RMSE across training runs:")
for arch, vm, vs, tm, ts, n in zip(
    ARCH_ORDER,
    stats_xyz["val_means"],
    stats_xyz["val_stds"],
    stats_xyz["test_means"],
    stats_xyz["test_stds"],
    stats_xyz["n_runs_list"],
):
    print(
        f"  {arch:8s}  val mean={vm:.4f}  sd={vs:.4f}  "
        f"test mean={tm:.4f}  sd={ts:.4f}  n_runs={n}"
    )


### M8 Step 4. Split sensitivity on `(x, y, z)`

Repeat the xyz train → validation / test protocol for `N_SENSITIVITY` independent particle-level splits (`SENSITIVITY_SPLIT_SEEDS`, default 60–64). Each split uses **one** training run (fixed training seed), so the spread reflects split variability rather than optimizer seeding. Catalog rows are relabeled in memory; the live workbook split is left unchanged.

This step answers the question of whether the performance advantage of the Fourier layer with relative to pooling, approaching flattening, can be generalized in the face of possible split randomization variability. The design of the study still means that the conclusion applies to this particular M8/M9 dataset, but that at least is not unique to the default split.

In terms of study design, step 2 and 3 are authoritative because they use the main, predefined split, this step is supportive as split is re-randomized.


In [ ]:
study_sens = run_split_sensitivity_study(
    catalog_rows=study_catalog_rows,
    task=localization_task_xyz,
    device=device,
    results_dir=RESULTS_DIR_SENS,
    lr_by_arch=LR_BY_ARCH,
    split_seeds=SENSITIVITY_SPLIT_SEEDS,
    num_epochs=NUM_EPOCHS_FULL,
    batch_size=BATCH_SIZE,
    training_seed=BASE_SEED,
    notebook_id="08_3A_2",
    experiment_id="m08_3a2_xyz_split_sensitivity",
    variant_prefix="m08_3a2xyz_sens",
    architecture_note_prefix="M8 xyz split-sensitivity",
    checkpoint_name=M08_XYZ_SPLIT_SENSITIVITY if _M8_CKPT.enabled else None,
    load_existing=_M8_CKPT.load_existing,
    retrain=_M8_CKPT.retrain,
)
print(f"study_sens skipped_train={study_sens.skipped_train}  checkpoint={display_path(study_sens.checkpoint_path)}")

print(f"split_seeds={study_sens.split_seeds}")
print(f"Wrote {repo_relative_path(study_sens.per_seed_path)}")
print(f"Wrote {repo_relative_path(study_sens.summary_path)}")
print(study_sens.per_seed_metrics.to_string(index=False))
print()
print(study_sens.summary_df.to_string(index=False))

fig, stats_sens = plot_rmse_summary_bars(
    study_sens.summary_df,
    variant_prefix="m08_3a2xyz_sens",
)
show_ml(
    fig,
    heading="M8 Step 4 — Split sensitivity",
    caption=(
        "xyz split-sensitivity validation / test RMSE "
        f"(mean ± SD across {stats_sens['n_runs']} splits)"
    ),
)

print("xyz split-sensitivity RMSE across split seeds:")
for arch, vm, vs, tm, ts, n in zip(
    ARCH_ORDER,
    stats_sens["val_means"],
    stats_sens["val_stds"],
    stats_sens["test_means"],
    stats_sens["test_stds"],
    stats_sens["n_runs_list"],
):
    print(
        f"  {arch:8s}  val mean={vm:.4f}  sd={vs:.4f}  "
        f"test mean={tm:.4f}  sd={ts:.4f}  n_splits={n}"
    )


### Conclusions from M8

- Average pooling performs worse than the other architectures, in z-localization and in 3D localization. Fourier performs substantially better, and usually (dependent on experimental fluctuations from run to run), Flatten performs best.
- This is interpreted to be related to Fourier-coded pooling and Flatten both preserving spatial information and achieving much lower validation / test error on both z and on xyz. Although not proof of the project hypothesis (you can never prove exactly a hypothesis) it supports the project hypothesis.
- It is also noteworthy that Fourier achieves comparable held-out performance with orders of magnitude fewer learned parameters (32k vs. 134M, e.g. a factor of about 4000).
- The split-sensitivity panel reports how those xyz conclusions hold under `N_SENSITIVITY` independent particle-level train/val/test partitions (one training seed each - only the partition is independent, the dataset is the same).

The split sensitivity analysis merits particular discussion.

The dataset intentionally includes particle locations spanning a wide range of localization difficulty, including near-null observations and multiple optical turbidity regimes. Because train, validation, and test partitions are randomized at the particle level, different partitions may contain different proportions of these difficult cases. This likely contributes to the observed variation in absolute RMSE between splits.

Despite this challenge, the qualitative architecture ranking was stable across the examined splits. Global average pooling consistently produced the highest error, while Fourier-coded and flatten-based readouts remained substantially better. Thus, although the absolute performance estimates vary with split composition, sometimes to a considerable degree, the central conclusion is robust: 

**The chosen method of preserving spatial information in the readout improves localization performance on single views.**


## Multi-view localization (M9)

The next step is to assess the benefit of the Fourier pooling layer in the context of view fusion for particle localization.

M8 established utility of Fourier encoding in a small network configuration for the analysis of single camera view.

M9 analyses whether Fourier encodings would be similarly useful in a multiview setting. Specifically, the views of a camera orbit around the Gummybear are analysed jointly.

The primary interest is to see whether the advantage of Fourier pooling over GAP (global average pooling) is conserved.

| Post-CNN pooling | Description |
|-------|-------------|
| GAP | Per channel, the feature map is averaged |
| Fourier| Per channel, the feature map is multiplied element-wise by Fourier terms, and averaged |

In order to understand generalizability, GAP vs. Fourier is tested in ladder of view embedding fusion methods with increasing complexity:

| View fusion method | Description |
|-------|-------------|
| single-view ref | Take embedding of the fixed 180° camera view (or mid-orbit if 180° absent), MLP to xyz |
| xyz-mean| MLP to xyz for every view, average x,y,z values |
| mean-pool| Average embeddings over views, then fusion MLP to x,y,z values |
| "DeepSets"| Permutation invariant fusion MLP: Linear(+ReLU) -> average -> MLP to x,y,z values |
| ordered-concat | Concatenate all embeddings, then fusion MLP to x,y,z values |

The idea here is to understand whether increased sophistication of the fusion layer interacts with the Fourier encoding, for example, making them more or less advantageous over GAP.

**Outputs shown:** Illustrative learning-rate study (reported Stage-B runs use historical defaults), val and test comparison of Fourier vs. GAP across the fusion methods.

**Training regime:** We also specifically compare separate training of CNN and fusion heads vs. joint end-to-end training. Indeed, end-to-end training might permit back-propagating gradients from the MLP to condition CNN to better exploit spatial information or better provide suitable Fourier embeddings, altering the balance between Fourier and GAP.

Approach:

| Step | Description | Test |
|-------|-------------|-----|
| Step 1 | Sequential scheme: train encoder, then fusion heads with frozen encoder | Fourier vs. Pooling on 5 fusion architecture variants |
| Step 2 | End-to-end training of the full network | Fourier vs. Pooling on compact / large fusion heads |

Note: The reason to include a "DeepSet" inspired architecture is that DeepSets ([Zaheer et al., 2017](https://arxiv.org/abs/1703.06114)) is considered a powerful middle-ground solution for image fusion while remaining order-agnostic.

Also note: Technically, the implementation uses rebatching (i.e. `views.reshape(batch * n_views, ...`, not directly parallel processing.


### M9 dataflow

The diagram below shows the dataflow in M9. The input views are from the common M8/M9 optical simulation dataset.


```mermaid
flowchart TD
  A["input views<br/>[V,C=1,H,W]"]

  subgraph ENC["parallel map over views: same encoder applied to each view"]
    direction LR

    V0["view 0<br/>[C=1,H,W]"]
    V1["view 1<br/>[C=1,H,W]"]
    VV["... view V-1<br/>[C=1,H,W]"]

    V0 --> CNN0["shared CNN<br/>feature map"]
    V1 --> CNN1["shared CNN<br/>feature map"]
    VV --> CNNV["shared CNN<br/>feature map"]

    CNN0 --> P0["Fourier or GAP<br/>pooling"]
    CNN1 --> P1["Fourier or GAP<br/>pooling"]
    CNNV --> PV["Fourier or GAP<br/>pooling"]

    P0 --> E0["[64] → Linear → [128]"]
    P1 --> E1["[64] → Linear → [128]"]
    PV --> EV["[64] → Linear → [128]"]
  end

  A --> V0
  A --> V1
  A --> VV

  E0 --> S["encoded view stack<br/>[V,128]"]
  E1 --> S
  EV --> S

  subgraph VAR["M9 compared variants: choose exactly one"]
    direction LR

    B1["single-view ref<br/>180° → xyz"]
    B2["xyz mean<br/>mean over view-wise xyz"]
    F1["mean-pool<br/>[V,128] → [128]"]
    F2["ordered concat<br/>[V,128] → [V·128]"]
    F3["DeepSets<br/>φ / ρ"]
  end

  S -. "variant 1" .-> B1
  S -. "variant 2" .-> B2
  S -. "variant 3" .-> F1
  S -. "variant 4" .-> F2
  S -. "variant 5" .-> F3

  B1 --> O1["xyz<br/>[3]"]
  B2 --> O2["xyz<br/>[3]"]

  F1 --> MLP["fusion MLP"]
  F2 --> MLP
  F3 --> MLP

  MLP --> O3["xyz<br/>[3]"]

  %% Styling
  classDef neutral fill:#242424,stroke:#2f6690,stroke-width:1.5px,color:#d6d6d6,font-weight:normal;
  classDef readout fill:#2f7145,stroke:#a7f3a7,stroke-width:4px,color:#ffffff,font-weight:bold,font-size:18px;
  classDef variant fill:#1f5f99,stroke:#93c5fd,stroke-width:4px,color:#ffffff,font-weight:bold,font-size:18px;

  class A,V0,V1,VV,CNN0,CNN1,CNNV,E0,E1,EV,S,MLP,O1,O2,O3 neutral;
  class P0,P1,PV readout;
  class B1,B2,F1,F2,F3 variant;
```




### M9 Step 1: Frozen Encoder

In this part, we train the M9 network separately for the CNN / MLP -> xyz per view, freeze the weights, and then train the fusion layer (where trainable, i.e. variant 3,4,5).

The question is whether the advantage of the Fourier-modulated embeddings observed in M8 over GAP pooling persists across multi-view fusion, first in the case of separately trained encoder and fusion head.

Reported Stage-B runs use historical learning-rate defaults; any LR sweep shown is illustrative only.


#### M9 Step 1 Setup

In [ ]:
from tomography_ml import get_device
from tomography_ml.studies import (
    study_checkpoint_policy,
    study_results_dir,
    DEFAULT_LR_STAGE_B,
    DEFAULT_LR_STAGE_B_GRID,
    M9FusionConfig,
    run_m9_frozen_fusion_family,
)
from tomography_ml.localization.builders import m8_single_view_block_freeze
from tomography_ml_validation.plotting import (
    combine_m9_comparisons,
    plot_m9_lr_study,
    plot_m9_param_counts_fourier_vs_pooled,
    plot_m9_rmse_fourier_vs_pooled,
    plot_m9_rmse_ladder,
)

# Same M8 single-particle corpus as notebooks/09_1A and 09_1B.
_M8_DEMO_WORKBOOK = ROOT / "configs" / "m8" / "m8_demo.xlsx"
_M8_FULL_WORKBOOK = ROOT / "configs" / "m8" / "localization_single_particle.xlsx"
_M8_DEMO_DATA_ROOT = ROOT / "data" / "generated" / "m8_demo"
_M8_FULL_DATA_ROOT = ROOT / "data" / "generated" / "m8_1" / "single_particle"
_M8_1_ROOT = ROOT / "data" / "generated" / "m8_1"

if DATA_MODE in ("inspect", "demo"):
    M9_WORKBOOK_PATH = _M8_DEMO_WORKBOOK
    M9_DATA_ROOT = _M8_DEMO_DATA_ROOT
    if not M9_WORKBOOK_PATH.is_file():
        M9_WORKBOOK_PATH = _copy_demo_workbook(M9_WORKBOOK_PATH)
    M9_RESULTS_ROOT = M9_DATA_ROOT
    # Short schedule for interactive demos.
    M9_NUM_EPOCHS = 40
    M9_EARLY_STOP = 25
    M9_ANGLE_STRIDE_DEG = 90
    M9_LR_GRID = (1e-2, 3e-3, 1e-3)
    M9_RUN_LR_STUDY = True
else:
    M9_WORKBOOK_PATH = _M8_FULL_WORKBOOK
    M9_DATA_ROOT = _M8_FULL_DATA_ROOT
    M9_RESULTS_ROOT = _M8_1_ROOT
    block = m8_single_view_block_freeze()
    M9_NUM_EPOCHS = int(block.num_epochs_max)  # 200
    M9_EARLY_STOP = int(block.early_stop_patience)  # 40
    M9_ANGLE_STRIDE_DEG = 60
    M9_LR_GRID = DEFAULT_LR_STAGE_B_GRID
    M9_RUN_LR_STUDY = True

_M9_CKPT = study_checkpoint_policy(
    repo_root=ROOT,
    milestone="m9",
    data_mode=DATA_MODE,
    read_checkpoints=READ_CHECKPOINTS,
)
RESULTS_DIR_M9A = study_results_dir(
    _M9_CKPT, fallback=M9_RESULTS_ROOT / "_09_1a_frozen_fourier_fusion"
)
RESULTS_DIR_M9B = study_results_dir(
    _M9_CKPT, fallback=M9_RESULTS_ROOT / "_09_1b_frozen_pooled_fusion"
)
LOAD_EXISTING_M9 = _M9_CKPT.load_existing
RETRAIN_M9 = _M9_CKPT.retrain

M9_BATCH_SIZE = int(m8_single_view_block_freeze().batch_size)  # 16
M9_LR_STAGE_B_DEFAULT = DEFAULT_LR_STAGE_B  # 3e-3
device_m9 = get_device()

print(f"DATA_MODE={DATA_MODE}")
print(f"M9_WORKBOOK_PATH={display_path(M9_WORKBOOK_PATH)}")
print(f"M9_DATA_ROOT={display_path(M9_DATA_ROOT)}")
print(f"RESULTS_DIR_M9A={display_path(RESULTS_DIR_M9A)}")
print(f"RESULTS_DIR_M9B={display_path(RESULTS_DIR_M9B)}")
print(
    f"epochs={M9_NUM_EPOCHS}  patience={M9_EARLY_STOP}  batch={M9_BATCH_SIZE}  "
    f"stride={M9_ANGLE_STRIDE_DEG}°  Stage-B default LR={M9_LR_STAGE_B_DEFAULT:g}  "
    f"RUN_LR_STUDY={M9_RUN_LR_STUDY}  RETRAIN_M9={RETRAIN_M9}"
)
print(f"Using device: {device_m9}")


#### M9 Step 1. Train Fourier and pooled fusion heads, two stage (frozen) mode

Stage A (single-view baseline encoder training) then Stage B (Frozen encoder from A + fusion heads).


In [ ]:
def _m9_cfg(family: str, results_dir):
    return M9FusionConfig(
        family=family,
        workbook_path=M9_WORKBOOK_PATH,
        data_root=M9_DATA_ROOT,
        results_dir=results_dir,
        stl_root=ROOT,
        device=device_m9,
        num_epochs=M9_NUM_EPOCHS,
        early_stop_patience=M9_EARLY_STOP,
        batch_size=M9_BATCH_SIZE,
        angle_stride_deg=M9_ANGLE_STRIDE_DEG,
        run_lr_study=M9_RUN_LR_STUDY,
        lr_stage_b_grid=M9_LR_GRID,
        lr_stage_b_mean_pool=M9_LR_STAGE_B_DEFAULT,
        lr_stage_b_ordered_concat=M9_LR_STAGE_B_DEFAULT,
        lr_stage_b_deepsets=M9_LR_STAGE_B_DEFAULT,
        compact_select_best_val_lr=False,
        deepsets_select_best_val_lr=False,  # LR sweep illustrative; Stage-B uses DEFAULT_LR_STAGE_B
        load_existing=LOAD_EXISTING_M9,
        retrain=RETRAIN_M9,
    )

print("=== M9 Fourier family  ===")
# run_m9_frozen_fusion_family runs the entire training and evaluation process. In detail:
# I)   train single view trunk. While training is like in M8, views from different angles are included
#        although no angle features or labels are provided
# II)  For each fusion head (mean-pool, DeepSets, ordered-concat), optional LR sweep
#        (illustrative; reported Stage-B models use historical DEFAULT_LR_STAGE_B) then train heads with encoder frozen
# III) Evaluate loss on val/test
# IV). Persist checkpoints and also human-readable output files. One pt file with multiple entries for the 
#        entire family.   
m9_fourier = run_m9_frozen_fusion_family(_m9_cfg("fourier", RESULTS_DIR_M9A))
print(
    f"views={m9_fourier.view_angles}  lr_stage_a={m9_fourier.lr_stage_a:g}  "
    f"selected_lrs={m9_fourier.selected_lrs}  skipped={m9_fourier.skipped_train}"
)
print(m9_fourier.comparison_df.to_string(index=False))

print("\n=== M9 pooled family (09_1B) ===")
m9_pooled = run_m9_frozen_fusion_family(_m9_cfg("pooled", RESULTS_DIR_M9B))
print(
    f"views={m9_pooled.view_angles}  lr_stage_a={m9_pooled.lr_stage_a:g}  "
    f"selected_lrs={m9_pooled.selected_lrs}  skipped={m9_pooled.skipped_train}"
)
print(m9_pooled.comparison_df.to_string(index=False))


#### M9 Step 1 Learning-rate curves, parameter counts, and validation / test RMSE

LR study is illustrative; reported Stage-B runs use historical default learning rates (illustrative LR sweeps are not used to select reported runs). Test-val final loss output on Fourier vs. Pooled, Fourier-vs-pooled parameter counts, then comparative Fourier-vs-pooled validation and test RMSE.

With these data, the step permits to address the question of whether the advantage of the Fourier-modulated embeddings observed in M8 persists across multi-view fusion. The design of the study means that this answer is limited to the case of the use of separately trained encoder and fusion heads. In particular, the xyz-averaging method is realistic when localization estimations from different views are collected and later pooled arithmetically.


In [ ]:
# LR studies (Stage B heads)
for family, result, title, heading in (
    ("fourier", m9_fourier, "09_1A Stage B LR studies (Fourier heads)", "M9 Step 1 — Fourier learning rates"),
    ("pooled", m9_pooled, "09_1B Stage B LR studies (pooled heads)", "M9 Step 1 — Pooled learning rates"),
):
    fig = plot_m9_lr_study(result.lr_study_df, family=family, title=title)
    if fig is None:
        print(f"No LR study table for {family} (fixed-LR only or missing CSV).")
    else:
        show_ml(fig, heading=heading, caption=title)

# Per-family validation / test ladders
fig = plot_m9_rmse_ladder(
    m9_fourier.comparison_df,
    family="fourier",
    title="09_1A Fourier fusion — total RMSE",
)
show_ml(fig, heading="M9 Step 1 — Fourier Val/Test", caption="09_1A Fourier fusion — total RMSE")
fig = plot_m9_rmse_ladder(
    m9_pooled.comparison_df,
    family="pooled",
    title="09_1B pooled fusion — total RMSE",
)
show_ml(fig, heading="M9 Step 1 — Pooled Val/Test", caption="09_1B pooled fusion — total RMSE")

# Parameter counts (Fourier vs pooled) — third plot in the 09_1B notebook sequence
fig = plot_m9_param_counts_fourier_vs_pooled(
    m9_fourier.comparison_df,
    m9_pooled.comparison_df,
)
if fig is None:
    print("Param-count comparison unavailable (missing learned_parameter_count).")
else:
    show_ml(
        fig,
        heading="M9 Step 1 — parameter counts",
        caption="09_1 trainable params — Fourier vs non-Fourier",
    )

# Fourier vs pooled validation / test
for split in ("validation", "test"):
    fig = plot_m9_rmse_fourier_vs_pooled(
        m9_fourier.comparison_df,
        m9_pooled.comparison_df,
        split=split,
    )
    if fig is None:
        print(f"No Fourier-vs-pooled RMSE pairs for {split}.")
    else:
        show_ml(
            fig,
            heading=f"M9 Step 1 — Fourier vs pooled ({split})",
            caption=f"09_1 Fourier vs pooled — {split}",
        )

combined_m9 = combine_m9_comparisons(
    m9_fourier.comparison_df, m9_pooled.comparison_df
)
out_combined = RESULTS_DIR_M9B / "m09_1_combined_fourier_vs_pooled.csv"
combined_m9.to_csv(out_combined, index=False)
print(f"Wrote {repo_relative_path(out_combined)}")
print(
    "Selected Stage-B LRs — "
    f"Fourier {m9_fourier.selected_lrs} | pooled {m9_pooled.selected_lrs}"
)


### Interpretation of M9 Step 1

- For the 2-step approach with a trained, then frozen encoder and a separately trained fusion head, the advantage of the Fourier embeddings persists across view fusion.
- However, the advantage decreases with increasing fusion head complexity. Apparently, a more complex fusion head permits to at least partially compensate for relative lack of spatial information in the embeddings.
- Note that the ladder confirms an advantage of the learned DeepSet-inspired fusion layer over simple pooling, in line with literature expectations. Fourier-encoding, in the particular setting of two-stage learning, provides an additional more minor advantage in the DeepSet setting.


### M9 Step 2: end-to-end geometry fusion

The aim here is to understand whether gradient descent into the CNN part in end to end learning helps to instruct the network to encode spatial information in ways independent or in competition with Fourier embeddings.

The network architectures compared are simplified and slightly adapted to this specific question:

| Fusion Head | Description |
| ----------- | ----------- |
| Single view | As in Step 1; not learned |
| xyz averaging | As in Step 1; not learned |
| Compact Head | 128-embedding, 1 Linear/ReLU block then project |
| Large Head | 512-embedding, 2 Linear/ReLU block then project |

Comparing a similarly constructed, but more or less expressive fusion head provides a simple measure on the impact of gradient descent in end to end training.



In [ ]:
from tomography_ml.studies import (
    M9E2EConfig,
    run_m9_e2e_geometry_fusion_family,
    study_results_dir,
)
from tomography_ml_validation.plotting import (
    combine_m9_comparisons,
    plot_m9_e2e_param_counts_fourier_vs_pooled,
    plot_m9_e2e_rmse_fourier_vs_pooled,
    plot_m9_e2e_rmse_ladder,
)

# Reuse M9 corpus / schedule / checkpoint policy from the frozen-setup cell above.
RESULTS_DIR_M9_E2E_A = study_results_dir(
    _M9_CKPT, fallback=M9_RESULTS_ROOT / "_09_2a_e2e_fourier_geometry_fusion"
)
RESULTS_DIR_M9_E2E_B = study_results_dir(
    _M9_CKPT, fallback=M9_RESULTS_ROOT / "_09_2b_e2e_pooled_geometry_fusion"
)
LOAD_EXISTING_M9_E2E = _M9_CKPT.load_existing
RETRAIN_M9_E2E = _M9_CKPT.retrain

print(f"RESULTS_DIR_M9_E2E_A={display_path(RESULTS_DIR_M9_E2E_A)}")
print(f"RESULTS_DIR_M9_E2E_B={display_path(RESULTS_DIR_M9_E2E_B)}")
print(
    f"epochs={M9_NUM_EPOCHS}  patience={M9_EARLY_STOP}  batch={M9_BATCH_SIZE}  "
    f"stride={M9_ANGLE_STRIDE_DEG}°  RETRAIN_M9_E2E={RETRAIN_M9_E2E}"
)


#### M9 Step 2: Train end-to-end Fourier and pooled ladders

End-to-end training with warm start on initial encoder (here, not prior weights), then full end-to-end training.


In [ ]:
def _m9_e2e_cfg(family: str, results_dir):
    return M9E2EConfig(
        family=family,
        workbook_path=M9_WORKBOOK_PATH,
        data_root=M9_DATA_ROOT,
        results_dir=results_dir,
        stl_root=ROOT,
        device=device_m9,
        num_epochs=M9_NUM_EPOCHS,
        early_stop_patience=M9_EARLY_STOP,
        batch_size=M9_BATCH_SIZE,
        angle_stride_deg=M9_ANGLE_STRIDE_DEG,
        load_existing=LOAD_EXISTING_M9_E2E,
        retrain=RETRAIN_M9_E2E,
    )

print("=== M9 e2e Fourier family (09_2A) ===")
# run_m9_e2e_geometry_fusion_family:
# I)  Stage A: train single-view trunk on multi-angle concat (same LR as frozen M9)
# II) Stage B: jointly train compact (09_2) and large (09_3) geometry-aware fusion
# III) Evaluate SV / xyz-mean / compact / large on val+test; persist checkpoint + CSV
m9_e2e_fourier = run_m9_e2e_geometry_fusion_family(
    _m9_e2e_cfg("fourier", RESULTS_DIR_M9_E2E_A)
)
print(
    f"views={m9_e2e_fourier.view_angles}  lr_stage_a={m9_e2e_fourier.lr_stage_a:g}  "
    f"skipped={m9_e2e_fourier.skipped_train}"
)
print(m9_e2e_fourier.comparison_df.to_string(index=False))

print("\n=== M9 e2e pooled family (09_2B) ===")
m9_e2e_pooled = run_m9_e2e_geometry_fusion_family(
    _m9_e2e_cfg("pooled", RESULTS_DIR_M9_E2E_B)
)
print(
    f"views={m9_e2e_pooled.view_angles}  lr_stage_a={m9_e2e_pooled.lr_stage_a:g}  "
    f"skipped={m9_e2e_pooled.skipped_train}"
)
print(m9_e2e_pooled.comparison_df.to_string(index=False))


#### M9 Step 2: Results
Comparison of Fourier vs. Pooling on validation and test split loss after end-to-end training.


In [ ]:
fig = plot_m9_e2e_rmse_ladder(
    m9_e2e_fourier.comparison_df,
    family="fourier",
    title="09_2A Fourier e2e + geometry — total RMSE",
)
show_ml(
    fig,
    heading="M9 Step 2 — Fourier RMSE",
    caption="09_2A Fourier e2e + geometry — total RMSE",
)
fig = plot_m9_e2e_rmse_ladder(
    m9_e2e_pooled.comparison_df,
    family="pooled",
    title="09_2B pooled e2e + geometry — total RMSE",
)
show_ml(
    fig,
    heading="M9 Step 2 — pooled RMSE",
    caption="09_2B pooled e2e + geometry — total RMSE",
)

fig = plot_m9_e2e_param_counts_fourier_vs_pooled(
    m9_e2e_fourier.comparison_df,
    m9_e2e_pooled.comparison_df,
)
if fig is None:
    print("Param-count comparison unavailable (missing learned_parameter_count).")
else:
    show_ml(
        fig,
        heading="M9 Step 2 — parameter counts",
        caption="09_2/09_3 trainable params — Fourier vs pooled",
    )

for split in ("validation", "test"):
    fig = plot_m9_e2e_rmse_fourier_vs_pooled(
        m9_e2e_fourier.comparison_df,
        m9_e2e_pooled.comparison_df,
        split=split,
    )
    if fig is None:
        print(f"No e2e Fourier-vs-pooled RMSE pairs for {split}.")
    else:
        show_ml(
            fig,
            heading=f"M9 Step 2 — Fourier vs pooled ({split})",
            caption=f"09_2/09_3 Fourier vs pooled — {split}",
        )

combined_m9_e2e = combine_m9_comparisons(
    m9_e2e_fourier.comparison_df, m9_e2e_pooled.comparison_df
)
out_combined_e2e = RESULTS_DIR_M9_E2E_B / "m09_e2e_combined_fourier_vs_pooled.csv"
combined_m9_e2e.to_csv(out_combined_e2e, index=False)
print(f"Wrote {repo_relative_path(out_combined_e2e)}")


#### Interpretation M9 step 2

- End-to-end training abolishes the beneficial effect of Fourier encoding in the multi-view camera setting.
- The advantage is still seen on the single view and xyz-mean controls which do not have a trainable fusion head. Therefore, the conclusion is specific to the use of fusion heads (this is not an algorithmic artifact of the evaluation).
- This comes with substantially larger models (albeit still moderate by modern standards).
- Scientifically, a plausible interpretation is that the gradient descent provides the CNN with the opportunity to encode information in the embeddings that are "retrievable" by the downstream, powerful MLP.
- Fourier-modulated embeddings might even be somewhat harmful. Possibly the higher frequency content modulation adds noise, possibly the gradient descent is able to find better encodings, possibly the small non-inferiority of the Fourier approach seen in the test split is statistical noise. Further analysis and experiment repetition would be needed to better understand this.


## Multi-illumination localization (M10)

M10 adds illumination as a source of physical information variation.

Compared to the addition of views, the addition of various illumination angles adds a major source of physical information: while camera views essentially complete the "hidden face" of a given gummybear physics, revolving the illumination source around the bear permits the particle to cast a revolving shadow and scatter halo in the gummybear.

The question for M10 is however the same as for M9: Does the advantage of Fourier embedding pooling survive through fusion of information from different views?

Based on the M9 results (which were not available at overall project design time, but are available at write-up): The question becomes rather whether illumination adds enough additional information so that Fourier embedding becomes unnecessary or harmful even the case of two step training between encoding and fusion.

| Protocol | Training | Fusion head variant |
|----------|----------|---------------------|
| Step 1 | Frozen: CNN Encoder, Fusion MLP separately | single-illumination / mean-xyz controls; compact ordered-concat heads C (no light angle) and D (same capacity, light-angle FiLM) |
| Step 2 | End-to-End: CNN Encoder and Fusion MLP jointly | same A–D ladder as Step 1 (C and D share compact capacity; D adds light-angle conditioning) |
| Step 3 | Hierarchical fusion: Illumination, then camera views | One defined fusion strategy, evaluated for Fourier and pooled: CNN per view (one illumination, one camera angle) -> MLP fuses illumination, using cos/sin illumination angle token -> MLP fuses camera views -> xyz |

Reported runs use historical / default learning rates rather than per-model optimal selection from LR sweeps.


### M10 Step 1 - separate encoder and illumination fusion training
#### Setup


In [ ]:
from pathlib import Path

from tomography_ml import get_device
from tomography_ml.localization.localize_multiview import M10_LIGHT_ANGLES_DEG
from tomography_ml.studies import (
    M10IlluminationConfig,
    run_m10_illumination_fusion,
    study_checkpoint_policy,
    study_results_dir,
    DEFAULT_LR_STAGE_B,
    DEFAULT_LR_STAGE_B_GRID,
)
from tomography_ml.localization.builders import m8_single_view_block_freeze
from tomography_ml_validation.plotting import (
    PLOT_CONFIG_10_1A,
    PLOT_CONFIG_10_1B,
    plot_illumination_fusion_results,
    plot_m10_backbone_rmse_ladder,
    plot_m10_param_counts_fourier_vs_pooled,
    plot_stage_b_lr_study,
)

# M10 corpus (not M8/M9): reuse workbook / lights from the M10 dataset section when present.
_M10_DEMO_ROOT = ROOT / "data" / "generated" / "m10_demo"
_M10_FULL_ROOT = ROOT / "data" / "generated" / "m10_illumination"
_M10_FULL_WB = ROOT / "configs" / "m10" / "localization_m10_illumination.xlsx"

if DATA_MODE in ("inspect", "demo"):
    M10_STUDY_DATA_ROOT = globals().get("OUTPUT_ROOT", _M10_DEMO_ROOT)
    M10_STUDY_WORKBOOK = globals().get("WORKBOOK_PATH", ROOT / "configs" / "m10" / "m10_demo.xlsx")
    if not Path(M10_STUDY_WORKBOOK).is_file() and "_copy_m10_demo_workbook" in globals():
        M10_STUDY_WORKBOOK = _copy_m10_demo_workbook(ROOT / "configs" / "m10" / "m10_demo.xlsx")
    M10_LIGHTS = tuple(globals().get("LIGHT_ANGLES_DEG", (0.0, 120.0, 240.0)))
    M10_NUM_EPOCHS = 40
    M10_EARLY_STOP = 25
    M10_N_REPEAT = 1
    M10_RUN_LR_FROZEN = False
    M10_RUN_LR_E2E = False
    M10_MIN_GROUPS = 2
else:
    M10_STUDY_DATA_ROOT = _M10_FULL_ROOT
    M10_STUDY_WORKBOOK = _M10_FULL_WB
    M10_LIGHTS = tuple(globals().get("LIGHT_ANGLES_DEG", M10_LIGHT_ANGLES_DEG))
    block_m10 = m8_single_view_block_freeze()
    M10_NUM_EPOCHS = int(block_m10.num_epochs_max)
    M10_EARLY_STOP = int(block_m10.early_stop_patience)
    M10_N_REPEAT = 3
    M10_RUN_LR_FROZEN = "if_unknown"
    M10_RUN_LR_E2E = False
    M10_MIN_GROUPS = 4

_M10_CKPT = study_checkpoint_policy(
    repo_root=ROOT,
    milestone="m10",
    data_mode=DATA_MODE,
    read_checkpoints=READ_CHECKPOINTS,
)
RESULTS_DIR_M10A = study_results_dir(
    _M10_CKPT, fallback=M10_STUDY_DATA_ROOT / "_10_1a_illumination_only_fusion"
)
RESULTS_DIR_M10B = study_results_dir(
    _M10_CKPT, fallback=M10_STUDY_DATA_ROOT / "_10_1b_illumination_only_fusion"
)
M10_BATCH_SIZE = int(m8_single_view_block_freeze().batch_size)
device_m10 = get_device()

print(f"DATA_MODE={DATA_MODE}")
print(f"M10_STUDY_WORKBOOK={display_path(M10_STUDY_WORKBOOK)}")
print(f"M10_STUDY_DATA_ROOT={display_path(M10_STUDY_DATA_ROOT)}")
print(f"RESULTS_DIR_M10A={display_path(RESULTS_DIR_M10A)}")
print(f"RESULTS_DIR_M10B={display_path(RESULTS_DIR_M10B)}")
print(
    f"lights={M10_LIGHTS}  epochs={M10_NUM_EPOCHS}  patience={M10_EARLY_STOP}  "
    f"batch={M10_BATCH_SIZE}  n_repeat={M10_N_REPEAT}  "
    f"RETRAIN={_M10_CKPT.retrain}"
)
print(f"Using device: {device_m10}")


### Train frozen illumination fusion (10_1A)

Stage A trains single-view trunks on all lights; Stage B freezes the encoder and trains fusion heads (A–D ladder: single-illumination / mean-xyz controls, then compact ordered-concat C without light angle and D with light-angle FiLM at the same capacity). Fourier and pooled variants are compared.


In [ ]:
def _m10_cfg(mode: str, results_dir, *, run_lr_study):
    return M10IlluminationConfig(
        mode=mode,
        workbook_path=Path(M10_STUDY_WORKBOOK),
        data_root=Path(M10_STUDY_DATA_ROOT),
        results_dir=results_dir,
        stl_root=ROOT,
        device=device_m10,
        num_epochs=M10_NUM_EPOCHS,
        early_stop_patience=M10_EARLY_STOP,
        batch_size=M10_BATCH_SIZE,
        light_angles_deg=M10_LIGHTS,
        min_joint_groups=M10_MIN_GROUPS,
        n_repeat_training=M10_N_REPEAT,
        run_lr_study=run_lr_study,
        lr_stage_b_grid=DEFAULT_LR_STAGE_B_GRID,
        lr_stage_b_c_fourier=DEFAULT_LR_STAGE_B,
        lr_stage_b_d_fourier=DEFAULT_LR_STAGE_B,
        lr_stage_b_c_pooled=DEFAULT_LR_STAGE_B,
        lr_stage_b_d_pooled=DEFAULT_LR_STAGE_B,
        select_best_val_lr=False,  # LR sweep illustrative; Stage-B uses DEFAULT_LR_STAGE_B
        load_existing=_M10_CKPT.load_existing,
        retrain=_M10_CKPT.retrain,
    )

print("=== M10 frozen (10_1A) ===")
m10_frozen = run_m10_illumination_fusion(
    _m10_cfg("frozen", RESULTS_DIR_M10A, run_lr_study=M10_RUN_LR_FROZEN)
)
print(f"skipped={m10_frozen.skipped_train}  runs={m10_frozen.session_run_ids}")
print(m10_frozen.comparison_df.to_string(index=False))


### M10 Step 1 - Single Camera View, Multi-illumination, Separate Encoder and Fusion Head Training
####Results: Learning rates, val/test performance, model size, Fourier vs. Pooled comparison

The models compared are:

- A Evaluation from illumination 0° at camera 180° per particle (xyz): "Single View reference"
- B Evaluation as the average of the xyz positions across all the illumination views "xyz mean"
- C Illumination fusion head (per illumination: CNN -> Fourier/GAP -> 64 -> Linear -> 128 concat I illumination embedding -> MLP (Linear 128*I -> 128, Relu, then Linear to xyz) -> 3), no illumination angle input
- D Same compact capacity as C, but with explicit light-angle conditioning (FiLM on the 128-d latents after encode, using cos(angle) and sin(angle))

Fourier vs. Pooled indicates utility of Four layer in each of these architectures. The reason for including explicit angle as learned feature-wise linear transformation (FiLM) is to understand whether the model needs explicit angular information or whether illumination cues and ordering are sufficient. 

Also note that while A is evaluated on a single view (illumination 0°, camera 180° per sample), training is on all illumination samples concatenated, so the physical information offered by the multi-illumination setting still enters model training.


In [ ]:
def _plot_m10_like_m9(result, plot_config, *, heading_prefix):
    _show = _ml_plt_show_restore(
        heading=f"{heading_prefix} — learning-rate study",
        caption=f"{plot_config.title_prefix} Stage B LR studies",
    )
    try:
        plot_stage_b_lr_study(
            results_dir=result.results_dir,
            config=plot_config,
            lr_study_df=result.lr_study_df if len(result.lr_study_df) else None,
            lr_study_path=result.lr_study_path,
        )
    finally:
        plt.show = _show
    summary = result.session_summary_df if len(result.session_summary_df) else None
    ladder_src = summary if summary is not None and len(summary) else result.comparison_df
    for kind, title, short in (
        ("fourier", f"{plot_config.title_prefix} — Fourier A–D RMSE", "Fourier RMSE"),
        ("pooled", f"{plot_config.title_prefix} — pooled A–D RMSE", "pooled RMSE"),
    ):
        fig = plot_m10_backbone_rmse_ladder(
            ladder_src, config=plot_config, backbone_kind=kind, title=title
        )
        if fig is None:
            print(f"No {kind} ladder for {plot_config.title_prefix}.")
        else:
            show_ml(fig, heading=f"{heading_prefix} — {short}", caption=title)
    fig = plot_m10_param_counts_fourier_vs_pooled(
        result.comparison_df, config=plot_config
    )
    if fig is None:
        print("Param-count comparison unavailable.")
    else:
        show_ml(
            fig,
            heading=f"{heading_prefix} — parameter counts",
            caption=f"{plot_config.title_prefix} trainable params — Fourier vs pooled",
        )
    _show = _ml_plt_show_restore(heading=f"{heading_prefix} — illumination fusion")
    try:
        plot_illumination_fusion_results(
            results_dir=result.results_dir,
            config=plot_config,
            history_path=result.history_path,
            session_summary_path=result.session_summary_path,
            comparison_path=result.comparison_path,
            lr_study_path=result.lr_study_path,
            session_run_ids=result.session_run_ids,
            session_summary_df=summary,
            comparison_df=result.comparison_df,
            lr_study_df=None,  # already plotted above
        )
    finally:
        plt.show = _show

_plot_m10_like_m9(m10_frozen, PLOT_CONFIG_10_1A, heading_prefix="M10 Step 1")


### M10 Step 1: Conclusions

- Fourier encoding remains useful, by a small margin, on single view evaluation (Model A).
- The fact that the margin is much smaller than in M8 or M9 with separate training is very interesting. A possible interpretation is that even though only a singly view is used as an input, training over multiple views enhanced the network's prediction capacity, particularly for the pooling architectures. This is in line with the general view that Fourier encodings are more useful in information-scarce than in information-riche prediction workflows, although the exact mechanism here is unclear.
- For the more complex fusion heads, and in fact even for simple xyz averaging, Fourier performs less well than the other models.
- In terms of fusion heads, the angle-aware head D seems slightly more performant than the angle-unaware head C at the same compact capacity.
- The conclusion is that Fourier is clearly not uniformly better: it remains a useful augmentation in the information-scarce single-view setting (Model A), but not when richer multi-illumination information is available for fusion.


### M10 Step 2: end-to-end illumination fusion

The question here is whether end-to-end training has a further impact on the relative performance of Fourier and GAP post-CNN pooling.

The same A–D model ladder is used, but with jointly trained trunks (e2e C/D). Checkpoint policy matches frozen M10.


In [ ]:
print("=== M10 end-to-end training (M10 step 2) ===")
m10_e2e = run_m10_illumination_fusion(
    _m10_cfg("e2e", RESULTS_DIR_M10B, run_lr_study=M10_RUN_LR_E2E)
)
print(f"skipped={m10_e2e.skipped_train}  runs={m10_e2e.session_run_ids}")
print(m10_e2e.comparison_df.to_string(index=False))


### M10 Step 2: Learning-rate curves, ladders, and Fourier vs pooled


In [ ]:
_plot_m10_like_m9(m10_e2e, PLOT_CONFIG_10_1B, heading_prefix="M10 Step 2")


### M10 Step 2: Conclusions and interpretation notes

The conclusion of step 2 (end-to-end training, N=3 repeat of the training) is that Fourier embedding is not advantageous, and in fact slightly detrimental on the fusion heads, confirming M10 Step 1 for the multi-illumination setting.

- Compare to 10_1A: does joint training change the Fourier vs pooled gap or the value of light∠ conditioning (D uses FiLM on 128-d latents, same compact capacity as C)?
- Stage-B LR sweeps (when enabled) are illustrative; reported models use the historical Stage-B default LR `3e-3`.
- Checkpoints: `m10_frozen_illumination_fusion.pt` / `m10_e2e_illumination_fusion.pt` under `checkpoints/m10/` in full mode.


### M10 Step 3: Hierarchical light-then-camera fusion

Steps 1–2 fixed the camera at 180° and fused illumination only (`[I,C,H,W]`). Step 3 uses the full illumination×camera grid with the preferred inductive bias: **fuse lights within each camera, then fuse cameras** (`for_10_2`, illumination-major `[I,V,C,H,W]`).

The ladder compares single-view reference, shared xyz-mean, and hierarchical fusion for Fourier and pooled backbones on validation/test. Stage A warm-starts single-view trunks; Stage B trains compact e2e hierarchical heads. Checkpoints: `m10_hierarchical_light_then_camera.pt` under `checkpoints/m10/` in full mode.


In [ ]:
from tomography_ml.studies.m10_hierarchical_fusion import (
    M10HierarchicalConfig,
    run_m10_hierarchical_fusion,
)

RESULTS_DIR_M10_STEP3 = study_results_dir(
    _M10_CKPT, fallback=M10_STUDY_DATA_ROOT / "_10_2_hierarchical_light_then_camera"
)
if DATA_MODE in ("inspect", "demo"):
    M10_HIER_STRIDE = 90.0
    M10_HIER_BATCH = 2
    M10_HIER_RUN_LR = True
else:
    M10_HIER_STRIDE = 10.0
    M10_HIER_BATCH = 1
    M10_HIER_RUN_LR = False

print("=== M10 hierarchical light-then-camera (Step 3) ===")
print(f"results={display_path(RESULTS_DIR_M10_STEP3)}")
print(
    f"stride={M10_HIER_STRIDE}  batch={M10_HIER_BATCH}  "
    f"run_lr_study={M10_HIER_RUN_LR}  load_existing={_M10_CKPT.load_existing}"
)

m10_hierarchical = run_m10_hierarchical_fusion(
    M10HierarchicalConfig(
        workbook_path=Path(M10_STUDY_WORKBOOK),
        data_root=Path(M10_STUDY_DATA_ROOT),
        results_dir=RESULTS_DIR_M10_STEP3,
        stl_root=ROOT,
        device=device_m10,
        num_epochs=M10_NUM_EPOCHS,
        early_stop_patience=M10_EARLY_STOP,
        batch_size=M10_HIER_BATCH,
        angle_stride_deg=M10_HIER_STRIDE,
        light_angles_deg=M10_LIGHTS,
        min_joint_groups=M10_MIN_GROUPS,
        run_lr_study=M10_HIER_RUN_LR,
        load_existing=_M10_CKPT.load_existing,
        retrain=_M10_CKPT.retrain,
    )
)
print(f"skipped={m10_hierarchical.skipped_train}")
print(m10_hierarchical.comparison_df.to_string(index=False))


### M10 Step 3: Results


In [ ]:
from tomography_ml_validation.plotting import (
    plot_m10_hierarchical_lr_study,
    plot_m10_hierarchical_rmse_fourier_vs_pooled,
)

if len(m10_hierarchical.lr_study_df):
    fig = plot_m10_hierarchical_lr_study(m10_hierarchical.lr_study_df)
    if fig is not None:
        show_ml(
            fig,
            heading="M10 Step 3 — Stage B LR study",
            caption="M10 Step 3 Stage B LR — Fourier vs pooled",
        )
else:
    print("LR study has single-LR rows only (no sweep).")

fig = plot_m10_hierarchical_rmse_fourier_vs_pooled(
    m10_hierarchical.comparison_df,
    title="M10 Step 3 hierarchical — Fourier vs pooled",
)
if fig is not None:
    show_ml(
        fig,
        heading="M10 Step 3 — hierarchical RMSE",
        caption="SV / xyz-mean / hierarchical — Fourier vs pooled",
    )


### M10 Step 3: Conclusions

- Hierarchical light-then-camera fusion completes the M10 ladder: structured acquisition axes (illumination, then camera) rather than treating lights as interchangeable views.
- Compare hierarchical heads to Steps 1–2 at fixed camera: does joint camera×light fusion recover information beyond illumination-only fusion?
- Stage-B LR sweeps (when enabled in demo mode) are illustrative; reported full-corpus models use historical Stage-B LR `3e-4`.
- Checkpoint: `m10_hierarchical_light_then_camera.pt` alongside `m10_2_comparison.csv` under `checkpoints/m10/`.


# Overall project conclusion

This project investigated whether physically informed Fourier-based spatial representations can improve particle localisation in translucent media while maintaining low model complexity. To address this question, a complete synthetic optical tomography framework was developed, enabling controlled evaluation of localisation architectures under single-view, multi-view, and multi-illumination conditions.

The results show that preserving spatial information is critical for accurate localisation. In the single-view setting (M8)—information-scarce because ML consumes a single camera angle under fixed illumination—Fourier pooling substantially outperformed Global Average Pooling while using orders of magnitude fewer parameters than flatten-based approaches. This effect remained valid across multiple train/validation/test partitions: absolute error varied with split composition, but the overall ranking remained stable: architectures that preserved spatial information consistently performed better than aggressive spatial averaging.

At the same time, the benefit of Fourier representations was not universal. In multi-view (M9) and multi-illumination (M10) experiments, their advantage decreased as additional observations and more expressive fusion models became available. Under end-to-end training, Fourier pooling often became neutral or slightly detrimental, suggesting that sufficiently powerful networks can learn alternative spatial encodings directly from the data.

The project hypothesis is essentially supported but augmented. Fourier-inspired spatial representations provide a valuable and highly parameter-efficient way to preserve localisation information when observations are limited. Their usefulness decreases as physical information content and model capacity increase. A further learning was that Fourier terms that were beneficial at low parameter count were also somewhat harmful for larger models with richer information access. Presumably it is preferable for the model to shape the embedding in its own way in these richer cases; the Fourier representation may add information limits or undesirable representation bias.

Beyond the specific machine-learning results, the project demonstrates a reproducible framework combining optical simulation, synthetic dataset generation, and deep-learning evaluation. Future work could for instance investigate multi-particle scenarios.

In summary, the central finding of this work is that explicit preservation of spatial information with Fourier embedding multipliers improves localisation performance under information-constrained conditions, while richer observations progressively reduce the need for handcrafted spatial-frequency priors.

### Software verification

The repository unit-test suite (`pytest`) checks catalog contracts, workbook randomization, and ML dataset loading used throughout this report. The cell below runs that suite against the same installed packages and local CAD/config assets as the notebook environment.


In [ ]:

!pytest -q --tb=line --color=yes

